# CARD Dataset External Validation

This notebook preprocesses the CARD dataset to match the C4 training feature space and validates the original C4-trained models (with SPQ).

**Key differences from YBT:**
- CARD has SPQ data (unlike YBT)
- CARD has one row per questionnaire entry (not per participant)
- Itemized scores are stored as CSV strings in the 'itemised score' column
- Must aggregate multiple questionnaires per participant

**Critical requirements:**
- Parse CSV itemized scores
- Aggregate questionnaires per participant (by VolunteerID)
- Create exactly 45 features matching C4 schema
- Exclude AQ features (data leakage prevention)
- Use original C4 scaler (NO refitting)
- Preserve original models (with SPQ) for CARD validation

**Execution order:** Run cells in order 1 -> 11 -> 6 -> 7 -> 8 -> 9. After Step 11 (AQ filter + balance), re-run Steps 6-7 so features match the filtered dataset, then 8-9 for validation.

In [ ]:
# Imports and configuration
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# Paths
ARTIFACT_DIR = '/Users/eb2007/playground/bullpy/c4_play2/models/cross_validation'
FEATURE_INFO_PATH = os.path.join(ARTIFACT_DIR, 'feature_info_original.json')
SCALER_PATH = os.path.join(ARTIFACT_DIR, 'scaler_original.joblib')
MODELS = {
    'Logistic Regression': os.path.join(ARTIFACT_DIR, 'logistic_regression_original.joblib'),
    'Random Forest': os.path.join(ARTIFACT_DIR, 'random_forest_original.joblib'),
    'XGBoost': os.path.join(ARTIFACT_DIR, 'xgboost_original.joblib'),
    'LightGBM': os.path.join(ARTIFACT_DIR, 'lightgbm_original.joblib'),
    'Gradient Boosting': os.path.join(ARTIFACT_DIR, 'gradient_boosting_original.joblib'),
}

# CARD dataset path - check both CSV and Excel versions
CARD_PATH_CSV = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/CARD_Nov2025(Sheet1).csv'
CARD_PATH_XLSX = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/CARD_Nov2025.xlsx'

# Use CSV if it exists, otherwise try Excel
import os
if os.path.exists(CARD_PATH_CSV):
    CARD_PATH = CARD_PATH_CSV
    print(f"Using CSV file: {CARD_PATH_CSV}")
elif os.path.exists(CARD_PATH_XLSX):
    CARD_PATH = CARD_PATH_XLSX
    print(f"Using Excel file: {CARD_PATH_XLSX}")
else:
    # Default to CSV path
    CARD_PATH = CARD_PATH_CSV
    print(f"Warning: File not found, will try: {CARD_PATH_CSV}")
EXCEL_PASSWORD = '£ddie4ever!'

OUTPUT_DIR = '/Users/eb2007/playground/bullpy/c4_play2/data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(42)

print("✅ Imports and paths configured")

## STEP 1: Load CARD Dataset

In [ ]:
print("="*80)
print("LOADING CARD DATASET")
print("="*80)

if CARD_PATH.lower().endswith(('.xlsx', '.xls')):
    try:
        df_card = pd.read_excel(CARD_PATH, engine='openpyxl')
        print("✅ Successfully loaded file (no password required)")
    except Exception as e:
        if 'BadZipFile' in str(type(e).__name__) or 'encrypted' in str(e).lower():
            print("⚠️  File appears to be password-protected.")
            print("Attempting to decrypt...")
            try:
                import msoffcrypto
                import io
                decrypted = io.BytesIO()
                with open(CARD_PATH, 'rb') as f:
                    office_file = msoffcrypto.OfficeFile(f)
                    office_file.load_key(password=EXCEL_PASSWORD)
                    office_file.decrypt(decrypted)
                    decrypted.seek(0)
                    df_card = pd.read_excel(decrypted, engine='openpyxl')
                print("✅ Successfully decrypted and loaded file")
            except ImportError:
                print("Installing msoffcrypto-tool...")
                import subprocess
                import sys
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'msoffcrypto-tool'])
                raise Exception("Please re-run this cell after installation")
        else:
            raise
else:
    # Try reading CSV with different encodings
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252']
    df_card = None
    for encoding in encodings:
        try:
            df_card = pd.read_csv(CARD_PATH, encoding=encoding)
            print(f"✅ Successfully loaded CSV file with {encoding} encoding")
            break
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"⚠️  Error reading with {encoding}: {e}")
            continue
    
    if df_card is None:
        raise ValueError(f"Could not read CSV file with any encoding. Tried: {encodings}")

print(f"\nCARD dataset shape (before aggregation): {df_card.shape}")
print(f"Columns: {list(df_card.columns)}")

# Identify key columns
volunteer_id_col = None
for col in df_card.columns:
    if 'volunteer' in col.lower() and 'id' in col.lower():
        volunteer_id_col = col
        break

if volunteer_id_col is None:
    for col in df_card.columns:
        if 'id' in col.lower() and df_card[col].nunique() < len(df_card) * 0.5:
            volunteer_id_col = col
            break

# Look for itemised score column (case-insensitive)
itemised_col = None

# First try exact matches by name (including "Itemised Score" with space)

# Explicit check for "Itemised Score" column (exact name)
if itemised_col is None:
    for col in df_card.columns:
        if col == "Itemised Score" or col == "itemised score" or col == "Itemized Score":
            itemised_col = col
            print(f"  ✓ Found itemised score column (exact match): '{col}'")
            break
for col in df_card.columns:
    col_lower = col.lower()
    if ('itemis' in col_lower or 'itemiz' in col_lower) and 'score' in col_lower:
        itemised_col = col
        print(f"  ✓ Found itemised score column: '{col}'")
        break

# If not found, search ALL columns for CSV data
if itemised_col is None:
    print("\n  Searching ALL columns for CSV itemized data...")
    for col in df_card.columns:
        sample_vals = df_card[col].dropna().head(5)
        if len(sample_vals) > 0:
            # Check multiple samples to be sure
            csv_count = 0
            for sample_val in sample_vals:
                sample_str = str(sample_val).strip()
                # Check if it looks like CSV with multiple items
                if ',' in sample_str:
                    num_items = len([x for x in sample_str.split(',') if x.strip()])
                    if num_items >= 10:  # Likely a full questionnaire
                        csv_count += 1
                        if csv_count >= 2:  # At least 2 rows have CSV data
                            itemised_col = col
                            print(f"  ✓ Found CSV data in column '{col}' ({num_items} items in sample)")
                            print(f"     Sample: {sample_str[:200]}...")
                            break
            if itemised_col:
                break

# Try case-insensitive search for column name variants (without spaces/dashes)
if itemised_col is None:
    for col in df_card.columns:
        col_clean = col.lower().replace(' ', '').replace('_', '').replace('-', '')
        if 'itemisedscore' in col_clean or 'itemizedscore' in col_clean:
            itemised_col = col
            print(f"  ✓ Found itemised score column (variant): '{col}'")
            break

# Debug: Show all columns that might be relevant
if itemised_col is None:
    print("\n  ⚠️  Could not find 'Itemised Score' column automatically.")
    print("  Columns containing 'score' or 'item':")
    for col in df_card.columns:
        col_lower = col.lower()
        if 'score' in col_lower or 'item' in col_lower:
            print(f"    - '{col}'")
            # Check if it contains CSV data
            sample_val = df_card[col].dropna().iloc[0] if len(df_card[col].dropna()) > 0 else None
            if sample_val is not None:
                sample_str = str(sample_val).strip()
                if ',' in sample_str:
                    num_items = len([x for x in sample_str.split(',') if x.strip()])
                    print(f"      → Contains CSV data ({num_items} items in sample)")
                else:
                    print(f"      → Single value: {sample_str[:50]}")

test_name_col = None
for col in df_card.columns:
    if 'test' in col.lower() or 'questionnaire' in col.lower() or ('name' in col.lower() and 'test' in col.lower()):
        test_name_col = col
        break

print(f"\nKey columns identified:")
print(f"  Volunteer ID: {volunteer_id_col}")
print(f"  Itemised score: {itemised_col}")
print(f"  Test name: {test_name_col}")

if volunteer_id_col:
    print(f"\nUnique participants: {df_card[volunteer_id_col].nunique()}")
    print(f"Total questionnaire entries: {len(df_card)}")
    print(f"Average entries per participant: {len(df_card) / df_card[volunteer_id_col].nunique():.2f}")

if test_name_col:
    print(f"\nUnique test types: {df_card[test_name_col].unique()[:10]}")

## STEP 2.5: Identify Item Mappings (HELPER CELL)

**CRITICAL**: Before parsing, we need to identify which items from the full questionnaires correspond to the 10-item short versions.

**To find the mappings:**
1. Check the Greenberg et al. (2018) PNAS paper supplementary materials
2. Or match item wording from the 10-item versions to positions in full questionnaires
3. Or check if CARD dataset has metadata about item positions

**Once you have the mappings**, update the `ITEM_MAPPINGS` dictionary in the next cell with the exact item positions (0-indexed).

## STEP 2: Parse Itemized Scores from CSV Column

In [ ]:
print("="*80)
print("PARSING ITEMIZED SCORES FROM CSV COLUMN")
print("="*80)

# ============================================================================
# ITEM MAPPING CONFIGURATION
# ============================================================================
# CRITICAL: The 10-item short versions use SPECIFIC SELECTED ITEMS from the 
# full questionnaires, NOT sequential items. These mappings need to be verified
# against the original papers or supplementary materials.
#
# Format: [item_position_in_full_questionnaire - 1] (0-indexed)
# Example: If AQ-10 item 1 corresponds to AQ-50 item 7, use [6] (0-indexed)
#
# Based on:
# - AQ-10: Allison et al. (2012) JAACAP
# - EQ-10, SQ-R-10, SPQ-10: Greenberg et al. (2018) PNAS supplementary materials
# ============================================================================

# Item mappings (0-indexed positions in full questionnaire)
ITEM_MAPPINGS = {
    'aq': {
        'full_length': 50,
        'short_items': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # AQ-10 uses items 1-10 from AQ-50 (Allison et al. 2012)
    },
    'eq': {
        'full_length': 60,
        'short_items': [13, 3, 8, 30, 27, 34, 11, 21, 17, 33]  # EQ-10 items: 14, 4, 9, 31, 28, 35, 12, 22, 18, 34 from EQ-60 (1-indexed)
    },
    'sqr': {
        'full_length': 75,
        'short_items': [31, 15, 26, 8, 29, 32, 11, 24, 7, 6]  # SQ-R-10 items: 32, 16, 27, 9, 30, 33, 12, 25, 8, 7 from SQ-R-75 (1-indexed)
    },
    'spq': {
        'full_length': 92,
        'short_items': [1, 20, 31, 34, 37, 57, 61, 72, 73, 87]  # SPQ-10 items: 2, 21, 32, 35, 38, 58, 62, 73, 74, 88 from SPQ-92 (1-indexed)
    }
}

print("\n✅ ITEM MAPPING STATUS:")
print("   Item mappings configured based on:")
print("   - EQ-10, SQ-R-10, SPQ-10: Greenberg et al. (2018) PNAS supplementary materials")
print("   - AQ-10: Allison et al. (2012) JAACAP")
print("\n   Mappings (0-indexed positions in full questionnaire):")
print(f"   - AQ-10: items {[x+1 for x in ITEM_MAPPINGS['aq']['short_items']]} from AQ-50")
print(f"   - EQ-10: items {[x+1 for x in ITEM_MAPPINGS['eq']['short_items']]} from EQ-60")
print(f"   - SQ-R-10: items {[x+1 for x in ITEM_MAPPINGS['sqr']['short_items']]} from SQ-R-75")
print(f"   - SPQ-10: items {[x+1 for x in ITEM_MAPPINGS['spq']['short_items']]} from SPQ-92")
print("="*80)

# Check if we found the columns (try case-insensitive if needed)
# NOTE: itemised_col should have been set in Cell 3, but check again here
if itemised_col is None:
    print("\n⚠️  WARNING: Could not find 'itemised score' column!")
    print("   Available columns:", list(df_card.columns))
    print("\n   Trying case-insensitive search...")
    
    # Comprehensive search: Check ALL columns for CSV data
    print("\n   Searching ALL columns for CSV itemized data...")
    for col in df_card.columns:
        sample_vals = df_card[col].dropna().head(3)
        if len(sample_vals) > 0:
            sample_str = str(sample_vals.iloc[0])
            # Check if it looks like CSV with multiple items
            if ',' in sample_str:
                num_items = len(sample_str.split(','))
                if num_items >= 10:  # Likely a full questionnaire
                    itemised_col = col
                    print(f"   ✓ Found CSV data in column '{col}' ({num_items} items in sample)")
                    print(f"      Sample: {sample_str[:200]}...")
                    break
    
        # Try case-insensitive search
    for col in df_card.columns:
        col_lower = col.lower()
        if ('itemis' in col_lower or 'itemiz' in col_lower) and 'score' in col_lower:
            itemised_col = col
            print(f"   ✓ Found itemised score column (case-insensitive): '{col}'")
            break
    
    # If still not found, check if 'Score' column contains CSV data
    if itemised_col is None and 'Score' in df_card.columns:
        print("\n   Checking if 'Score' column contains CSV itemized data...")
        sample_scores = df_card['Score'].dropna().head(10)
        csv_found = False
        for idx, val in enumerate(sample_scores):
            val_str = str(val)
            if ',' in val_str and len(val_str.split(',')) >= 5:  # Likely CSV if has multiple comma-separated values
                print(f"   ✓ Found CSV data in 'Score' column (sample row {idx+1})")
                print(f"      Sample: {val_str[:150]}...")
                csv_found = True
                itemised_col = 'Score'
                break
        
        if not csv_found:
            print("   'Score' column does not appear to contain CSV data")
            print("   Sample Score values:")
            for idx, val in enumerate(sample_scores.head(3)):
                print(f"     {idx+1}. {str(val)[:100]}...")
    
    if itemised_col is None:
        print("\n   ⚠️  Could not find itemized score column.")
        print("   Options:")
        print("   1. Itemized scores might be in 'Score' column (but not CSV format)")
        print("   2. Itemized scores might be in separate columns")
        print("   3. Column name might be different")
        print("\n   Please check the data structure and specify the column manually if needed.")
        # Don't raise error yet - let's see what we can do with Score column
        if 'Score' in df_card.columns:
            print("\n   ⚠️  CRITICAL: Using 'Score' column as fallback, but it contains totals not CSV!")
            print("   This will result in only 1 item per questionnaire instead of 10.")
            print("   Please check your Excel file for a column with CSV itemized scores.")
            itemised_col = 'Score'
        else:
            raise ValueError(f"Could not find itemized score column! Available columns: {list(df_card.columns)}")

if test_name_col is None:
    print("\n⚠️  WARNING: Could not find test name column!")
    print("   Available columns:", list(df_card.columns))
    if 'TestName' in df_card.columns:
        test_name_col = 'TestName'
        print(f"   Using 'TestName' column as fallback")
    else:
        raise ValueError("Could not find test name column! Please check column names.")

def parse_itemised_scores(itemised_str, test_name):
    """
    Parse CSV itemized scores and return as dictionary
    Handles both full questionnaires and short 10-item versions
    
    CRITICAL: The 10-item short versions use SPECIFIC SELECTED ITEMS from the full questionnaires,
    NOT the first 10 items sequentially. These items were chosen through psychometric validation.
    
    Item mappings needed (0-indexed positions in full questionnaire):
    - AQ-10: Specific items from AQ-50 (need exact mapping)
    - EQ-10: Specific items from EQ-60 (need exact mapping from Greenberg et al. 2018)
    - SQ-R-10: Specific items from SQ-R-75 (need exact mapping from Greenberg et al. 2018)
    - SPQ-10: Specific items from SPQ-92 (need exact mapping from Greenberg et al. 2018)
    
    References:
    - AQ-10: Allison, C., Auyeung, B., & Baron-Cohen, S. (2012). JAACAP, 51(2), 202-212
    - EQ-10, SQ-R-10, SPQ-10: Greenberg, D.M., et al. (2018). PNAS. DOI: 10.1073/pnas.1811032115
    """
    if pd.isna(itemised_str):
        return {}
    
    try:
        # Split by comma
        items = str(itemised_str).split(',')
        # Clean whitespace and convert to numeric
        items = [float(x.strip()) for x in items if x.strip() and x.strip().lower() != 'nan']
        
        if len(items) == 0:
            return {}
        
        # Create item dictionary based on test name
        item_dict = {}
        test_lower = str(test_name).lower()
        
        # Determine questionnaire type and apply correct mapping
        if 'aq' in test_lower and 'adolescent' not in test_lower and 'child' not in test_lower:
            # Adult AQ
            if len(items) == 10:
                # Already short version - use directly
                for i, val in enumerate(items, 1):
                    item_dict[f'aq_{i}'] = val
            elif len(items) >= ITEM_MAPPINGS['aq']['full_length']:
                # Full AQ-50 - apply mapping
                mapping = ITEM_MAPPINGS['aq']['short_items']
                if mapping is None:
                    print(f"  ERROR: AQ-10 item mapping not configured!")
                    print(f"         Found {len(items)} items but need exact mapping from AQ-50 to AQ-10")
                    print(f"         Please update ITEM_MAPPINGS['aq']['short_items'] with correct item positions")
                else:
                    # Apply mapping
                    for short_idx, full_idx in enumerate(mapping, 1):
                        if full_idx < len(items):
                            item_dict[f'aq_{short_idx}'] = items[full_idx]
            else:
                # Unknown length - assume already short or use what we have
                for i, val in enumerate(items[:10], 1):
                    item_dict[f'aq_{i}'] = val
                    
        elif 'eq' in test_lower and 'adolescent' not in test_lower and 'child' not in test_lower:
            # Adult EQ
            if len(items) == 10:
                # Already short version - use directly
                for i, val in enumerate(items, 1):
                    item_dict[f'eq_{i}'] = val
            elif len(items) >= ITEM_MAPPINGS['eq']['full_length']:
                # Full EQ-60 - apply mapping
                mapping = ITEM_MAPPINGS['eq']['short_items']
                if mapping is None:
                    print(f"  ERROR: EQ-10 item mapping not configured!")
                    print(f"         Found {len(items)} items but need exact mapping from EQ-60 to EQ-10")
                    print(f"         Please update ITEM_MAPPINGS['eq']['short_items'] with correct item positions")
                else:
                    # Apply mapping
                    for short_idx, full_idx in enumerate(mapping, 1):
                        if full_idx < len(items):
                            item_dict[f'eq_{short_idx}'] = items[full_idx]
            else:
                # Unknown length - assume already short or use what we have
                for i, val in enumerate(items[:10], 1):
                    item_dict[f'eq_{i}'] = val
                    
        elif ('sq' in test_lower or 'sqr' in test_lower) and 'adolescent' not in test_lower and 'child' not in test_lower:
            # Adult SQ/SQR
            if len(items) == 10:
                # Already short version - use directly
                for i, val in enumerate(items, 1):
                    item_dict[f'sqr_{i}'] = val
            elif len(items) >= ITEM_MAPPINGS['sqr']['full_length']:
                # Full SQ-R-75 - apply mapping
                mapping = ITEM_MAPPINGS['sqr']['short_items']
                if mapping is None:
                    print(f"  ERROR: SQ-R-10 item mapping not configured!")
                    print(f"         Found {len(items)} items but need exact mapping from SQ-R-75 to SQ-R-10")
                    print(f"         Please update ITEM_MAPPINGS['sqr']['short_items'] with correct item positions")
                else:
                    # Apply mapping
                    for short_idx, full_idx in enumerate(mapping, 1):
                        if full_idx < len(items):
                            item_dict[f'sqr_{short_idx}'] = items[full_idx]
            else:
                # Unknown length - assume already short or use what we have
                for i, val in enumerate(items[:10], 1):
                    item_dict[f'sqr_{i}'] = val
                    
        elif 'spq' in test_lower and 'child' not in test_lower:
            # Adult SPQ
            if len(items) == 10:
                # Already short version - use directly
                for i, val in enumerate(items, 1):
                    item_dict[f'spq_{i}'] = val
            elif len(items) >= ITEM_MAPPINGS['spq']['full_length']:
                # Full SPQ-92 - apply mapping
                mapping = ITEM_MAPPINGS['spq']['short_items']
                if mapping is None:
                    print(f"  ERROR: SPQ-10 item mapping not configured!")
                    print(f"         Found {len(items)} items but need exact mapping from SPQ-92 to SPQ-10")
                    print(f"         Please update ITEM_MAPPINGS['spq']['short_items'] with correct item positions")
                else:
                    # Apply mapping
                    for short_idx, full_idx in enumerate(mapping, 1):
                        if full_idx < len(items):
                            item_dict[f'spq_{short_idx}'] = items[full_idx]
            else:
                # Unknown length - assume already short or use what we have
                for i, val in enumerate(items[:10], 1):
                    item_dict[f'spq_{i}'] = val
        
        return item_dict
    except Exception as e:
        return {}

# Check if itemized scores are already in separate columns
print("\nChecking if itemized scores are in separate columns...")
existing_item_cols = {}
for col in df_card.columns:
    col_lower = col.lower()
    # Check for patterns like eq_1, aq_1, spq_1, sqr_1, sq_1
    for prefix in ['eq_', 'aq_', 'spq_', 'sqr_', 'sq_']:
        if col_lower.startswith(prefix) and col_lower[len(prefix):].isdigit():
            item_num = int(col_lower[len(prefix):])
            if 1 <= item_num <= 10:
                if prefix not in existing_item_cols:
                    existing_item_cols[prefix] = {}
                existing_item_cols[prefix][item_num] = col
                break

if existing_item_cols:
    print(f"   ✓ Found existing item columns:")
    for prefix, items in existing_item_cols.items():
        print(f"      {prefix}: {len(items)} items found")
    print("   Will use existing columns instead of parsing CSV")
    use_existing_columns = True
else:
    print("   No existing item columns found - will parse from CSV")
    use_existing_columns = False

# Parse itemized scores for each row
print("\nParsing itemized scores from CSV...")
parsed_items = []
item_count_stats = {}  # Track how many items were found per test type

if use_existing_columns:
    # If columns already exist, just copy them
    for idx, row in df_card.iterrows():
        parsed = {}
        test_name = str(row[test_name_col]).lower() if pd.notna(row[test_name_col]) else ''
        
        # Map test names to prefixes
        if 'aq' in test_name and 'adolescent' not in test_name and 'child' not in test_name:
            prefix = 'aq_'
        elif 'eq' in test_name and 'adolescent' not in test_name and 'child' not in test_name:
            prefix = 'eq_'
        elif ('sq' in test_name or 'sqr' in test_name) and 'adolescent' not in test_name and 'child' not in test_name:
            prefix = 'sqr_'
        elif 'spq' in test_name and 'child' not in test_name:
            prefix = 'spq_'
        else:
            prefix = None
        
        if prefix and prefix in existing_item_cols:
            for item_num, col_name in existing_item_cols[prefix].items():
                if pd.notna(row[col_name]):
                    parsed[f'{prefix.rstrip("_")}_{item_num}'] = row[col_name]
        
        parsed_items.append(parsed)
else:
    # Parse from CSV column
    for idx, row in df_card.iterrows():
        test_name = row[test_name_col] if pd.notna(row[test_name_col]) else ''
        itemised_str = row[itemised_col] if itemised_col else None
        
        # Count items in CSV for debugging
        if pd.notna(itemised_str):
            item_count = len(str(itemised_str).split(','))
            test_key = str(test_name).lower()
            if test_key not in item_count_stats:
                item_count_stats[test_key] = []
            item_count_stats[test_key].append(item_count)
        
        parsed = parse_itemised_scores(itemised_str, test_name)
        parsed_items.append(parsed)
    
    # Print statistics about item counts
    if item_count_stats:
        print("\nItem count statistics by test type:")
        for test_type, counts in item_count_stats.items():
            if counts:
                print(f"  {test_type}: {len(counts)} entries, item counts: min={min(counts)}, max={max(counts)}, median={np.median(counts):.0f}")

# Collect all unique item keys
all_item_keys = set()
for parsed in parsed_items:
    all_item_keys.update(parsed.keys())

# Initialize columns for parsed items
for item_key in all_item_keys:
    df_card[item_key] = np.nan

# Fill in parsed items
for idx, parsed in enumerate(parsed_items):
    for key, val in parsed.items():
        df_card.loc[idx, key] = val

print(f"\n✅ Parsed itemized scores")
print(f"   Created columns: {sorted(all_item_keys)}")
print(f"   Total item columns: {len(all_item_keys)}")

# Show how many rows have each item type
if all_item_keys:
    print(f"\nItem coverage:")
    for item_type in ['aq', 'eq', 'sqr', 'spq']:
        matching_items = [k for k in all_item_keys if k.startswith(f'{item_type}_')]
        if matching_items:
            matching_items_sorted = sorted(matching_items, key=lambda x: int(x.split('_')[1]))
            rows_with_data = df_card[matching_items_sorted].notna().any(axis=1).sum()
            print(f"  {item_type.upper()}: {len(matching_items)} items found, {rows_with_data} rows have data")


## STEP 3: Aggregate Multiple Questionnaires Per Participant

In [ ]:
print("="*80)
print("AGGREGATING MULTIPLE QUESTIONNAIRES PER PARTICIPANT")
print("="*80)

# Define questionnaire priority (adult > adolescent > child)
test_priority = {
    'aq': 1, 'adolescent aq': 2, 'child aq': 3,
    'eq': 1, 'adolescent eq': 2, 'child eq': 3, 'childeq': 3,
    'sq': 1, 'sqr': 1, 'adolescent sq': 2, 'child sq': 3, 'childsq': 3,
    'spq': 1, 'child_spq': 3
}

def get_priority(test_name):
    test_lower = str(test_name).lower()
    for key, priority in test_priority.items():
        if key in test_lower:
            return priority
    return 99  # Unknown tests get lowest priority

df_card['test_priority'] = df_card[test_name_col].apply(get_priority)

# Identify date column (if available)
date_col = None
for col in df_card.columns:
    if any(kw in col.lower() for kw in ['date', 'time', 'modified', 'created']):
        date_col = col
        break

if date_col:
    try:
        df_card[date_col] = pd.to_datetime(df_card[date_col], errors='coerce')
        df_card = df_card.sort_values(by=[volunteer_id_col, date_col])
        print(f"Found date column: '{date_col}' - sorted by date")
    except:
        date_col = None

# Aggregate: For each participant, collect best available questionnaire for each type
print("\nAggregating questionnaires per participant...")
participant_data = []

for volunteer_id in df_card[volunteer_id_col].unique():
    participant_rows = df_card[df_card[volunteer_id_col] == volunteer_id].copy()
    
    # Start with demographic info (take from first row)
    participant_row = participant_rows.iloc[0].copy()
    
    # Collect questionnaire items (prefer higher priority tests)
    questionnaire_items = {}
    
    # Group by test type and take highest priority
    for test_type in ['aq', 'eq', 'sq', 'sqr', 'spq']:
        matching_rows = participant_rows[
            participant_rows[test_name_col].astype(str).str.lower().str.contains(test_type, na=False)
        ]
        if len(matching_rows) > 0:
            # Take highest priority (lowest number)
            best_row = matching_rows.loc[matching_rows['test_priority'].idxmin()]
            
            # Extract items for this questionnaire
            if test_type == 'aq':
                for i in range(1, 11):
                    col = f'aq_{i}'
                    if col in best_row.index and pd.notna(best_row[col]):
                        questionnaire_items[col] = best_row[col]
            elif test_type == 'eq':
                for i in range(1, 11):
                    col = f'eq_{i}'
                    if col in best_row.index and pd.notna(best_row[col]):
                        questionnaire_items[col] = best_row[col]
            elif test_type in ['sq', 'sqr']:
                for i in range(1, 11):
                    col = f'sqr_{i}'
                    if col in best_row.index and pd.notna(best_row[col]):
                        questionnaire_items[col] = best_row[col]
            elif test_type == 'spq':
                for i in range(1, 11):
                    col = f'spq_{i}'
                    if col in best_row.index and pd.notna(best_row[col]):
                        questionnaire_items[col] = best_row[col]
    
    # Add questionnaire items to participant row
    for key, val in questionnaire_items.items():
        participant_row[key] = val
    
    participant_data.append(participant_row)

df_card_aggregated = pd.DataFrame(participant_data)

# Rename volunteer ID to userid
if volunteer_id_col != 'userid':
    df_card_aggregated = df_card_aggregated.rename(columns={volunteer_id_col: 'userid'})

print(f"\n✅ Aggregated dataset:")
print(f"   Participants: {len(df_card_aggregated)}")
print(f"   Columns: {len(df_card_aggregated.columns)}")

## STEP 4: Score Questionnaires (Matching C4 Rules)

In [ ]:
print("="*80)
print("SCORING QUESTIONNAIRES (MATCHING C4 RULES)")
print("="*80)

# SPQ-10 Scoring (Continuous 0-3 scale, range 0-30)
# C4 uses: 1->3, 2->2, 3->1, 4->0 (so: score = 4 - raw_value)
print("\nSPQ-10 Scoring...")
spq_cols = [f'spq_{i}' for i in range(1, 11)]
for col in spq_cols:
    if col in df_card_aggregated.columns:
        df_card_aggregated[col] = pd.to_numeric(df_card_aggregated[col], errors='coerce')
        # C4 scoring: if values are 1-4, convert to 0-3 scale
        if df_card_aggregated[col].notna().any():
            min_val = df_card_aggregated[col].min()
            max_val = df_card_aggregated[col].max()
            if min_val >= 1 and max_val <= 4:
                # Convert 1,2,3,4 to 3,2,1,0
                df_card_aggregated[col] = 4 - df_card_aggregated[col]

# Only sum columns that exist
existing_spq_cols = [col for col in spq_cols if col in df_card_aggregated.columns]
if existing_spq_cols:
    df_card_aggregated['spq_total'] = df_card_aggregated[existing_spq_cols].sum(axis=1)
    print(f"  ✅ SPQ-10 scored: {len(existing_spq_cols)} items found, total range {df_card_aggregated['spq_total'].min():.0f}-{df_card_aggregated['spq_total'].max():.0f}")
else:
    df_card_aggregated['spq_total'] = 0
    print(f"  ⚠️  No SPQ columns found - setting spq_total to 0")

# EQ-10 Scoring (Binary 0-1 with reverse-scoring)
print("\nEQ-10 Scoring...")
eq_cols = [f'eq_{i}' for i in range(1, 11)]
eq_reverse_items = [3]  # Item 3 is reverse-scored

for i in range(1, 11):
    col = f'eq_{i}'
    if col in df_card_aggregated.columns:
        df_card_aggregated[col] = pd.to_numeric(df_card_aggregated[col], errors='coerce')
        if i in eq_reverse_items:
            # Reverse: disagree (1,2) = 1, agree (3,4) = 0
            df_card_aggregated[col] = df_card_aggregated[col].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] 
                else 0 if pd.notna(x) and x in [3, 4] 
                else np.nan
            )
        else:
            # Normal: agree (3,4) = 1, disagree (1,2) = 0
            df_card_aggregated[col] = df_card_aggregated[col].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] 
                else 0 if pd.notna(x) and x in [1, 2] 
                else np.nan
            )

# Only sum columns that exist
existing_eq_cols = [col for col in eq_cols if col in df_card_aggregated.columns]
if existing_eq_cols:
    df_card_aggregated['eq_total'] = df_card_aggregated[existing_eq_cols].sum(axis=1)
    print(f"  ✅ EQ-10 scored: {len(existing_eq_cols)} items found, total range {df_card_aggregated['eq_total'].min():.0f}-{df_card_aggregated['eq_total'].max():.0f}")
else:
    df_card_aggregated['eq_total'] = 0
    print(f"  ⚠️  No EQ columns found - setting eq_total to 0")

# SQR-10 Scoring (Binary 0-1 with reverse-scoring)
print("\nSQR-10 Scoring...")
sqr_cols = [f'sqr_{i}' for i in range(1, 11)]
sqr_reverse_items = [2, 4, 6, 8, 10]

for i in range(1, 11):
    col = f'sqr_{i}'
    if col in df_card_aggregated.columns:
        df_card_aggregated[col] = pd.to_numeric(df_card_aggregated[col], errors='coerce')
        if i in sqr_reverse_items:
            # Reverse: disagree (1,2) = 1, agree (3,4) = 0
            df_card_aggregated[col] = df_card_aggregated[col].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] 
                else 0 if pd.notna(x) and x in [3, 4] 
                else np.nan
            )
        else:
            # Normal: agree (3,4) = 1, disagree (1,2) = 0
            df_card_aggregated[col] = df_card_aggregated[col].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] 
                else 0 if pd.notna(x) and x in [1, 2] 
                else np.nan
            )

# Only sum columns that exist
existing_sqr_cols = [col for col in sqr_cols if col in df_card_aggregated.columns]
if existing_sqr_cols:
    df_card_aggregated['sqr_total'] = df_card_aggregated[existing_sqr_cols].sum(axis=1)
    print(f"  ✅ SQR-10 scored: {len(existing_sqr_cols)} items found, total range {df_card_aggregated['sqr_total'].min():.0f}-{df_card_aggregated['sqr_total'].max():.0f}")
else:
    df_card_aggregated['sqr_total'] = 0
    print(f"  ⚠️  No SQR columns found - setting sqr_total to 0")

# AQ-10 Scoring (For reference only - will be excluded)
print("\nAQ-10 Scoring (for reference only - will be excluded)...")
aq_cols = [f'aq_{i}' for i in range(1, 11)]
aq_reverse_items = [2, 3, 4, 5, 6, 9]

for i in range(1, 11):
    col = f'aq_{i}'
    if col in df_card_aggregated.columns:
        df_card_aggregated[col] = pd.to_numeric(df_card_aggregated[col], errors='coerce')
        if i in aq_reverse_items:
            df_card_aggregated[col] = df_card_aggregated[col].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] 
                else 0 if pd.notna(x) and x in [3, 4] 
                else np.nan
            )
        else:
            df_card_aggregated[col] = df_card_aggregated[col].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] 
                else 0 if pd.notna(x) and x in [1, 2] 
                else np.nan
            )

# Only sum columns that exist
existing_aq_cols = [col for col in aq_cols if col in df_card_aggregated.columns]
if existing_aq_cols:
    df_card_aggregated['aq_total'] = df_card_aggregated[existing_aq_cols].sum(axis=1)
    print(f"  ✅ AQ-10 scored: {len(existing_aq_cols)} items found, total range {df_card_aggregated['aq_total'].min():.0f}-{df_card_aggregated['aq_total'].max():.0f}")
else:
    df_card_aggregated['aq_total'] = 0
    print(f"  ⚠️  No AQ columns found - setting aq_total to 0")
print(f"  ⚠️  NOTE: AQ features will be EXCLUDED from final feature set (data leakage prevention)")

## STEP 11: Create Balanced 50/50 Dataset

Create a balanced 50/50 dataset by:
1. **AQ threshold filtering** (matching C4 preprocessing): Remove ASC cases with AQ < 6
2. Filtering out adolescent/child questionnaires (C4 models were trained on adults only)
3. Downsampling to a true 50/50 split: min(cases, controls) in each class (e.g. 508:508 after AQ filter)

**Why AQ filtering?** C4 dataset excluded autism cases with AQ < 6 to create a cleaner, more consistent dataset. Matching this preprocessing should improve model performance on CARD.

In [ ]:
print("="*80)
print("CREATING BALANCED 50/50 DATASET (DOWNSAMPLING)")
print("="*80)

from sklearn.utils import resample

# Capture original size BEFORE any filtering (for summary)
original_size = len(df_card_aggregated)

# Check if autism_target exists, if not create it (in case Step 5 hasn't been run)
if 'autism_target' not in df_card_aggregated.columns:
    print("\n⚠️  'autism_target' not found - creating it from diagnosis column...")
    diagnosis_cols = [c for c in df_card_aggregated.columns if 'diagnos' in c.lower() or 'asc' in c.lower()]
    if diagnosis_cols:
        diagnosis_col = diagnosis_cols[0]
        print(f"  Using diagnosis column: '{diagnosis_col}'")
        diagnosis_series = df_card_aggregated[diagnosis_col]
        
        # Try different formats
        if diagnosis_series.dtype in ['int64', 'float64']:
            df_card_aggregated['autism_target'] = (diagnosis_series == 1).astype(int)
        else:
            diagnosis_numeric = pd.to_numeric(diagnosis_series, errors='coerce')
            if diagnosis_numeric.notna().any() and diagnosis_numeric.nunique() <= 2:
                df_card_aggregated['autism_target'] = (diagnosis_numeric == 1).astype(int)
            else:
                diagnosis_str = diagnosis_series.astype(str).str.lower()
                df_card_aggregated['autism_target'] = (
                    diagnosis_str.str.contains('autism|asc|asd|yes|1|true', case=False, na=False) &
                    ~diagnosis_str.str.contains('no|none|0|false|na|nan', case=False, na=False)
                ).astype(int)
        print(f"  ✅ Created autism_target: {df_card_aggregated['autism_target'].value_counts().to_dict()}")
    else:
        raise ValueError("Cannot find diagnosis column to create autism_target. Please run Step 5 first.")

# STEP 0: Apply AQ threshold filtering (matching C4 preprocessing)
print("\n" + "="*80)
print("STEP 0: APPLYING AQ THRESHOLD FILTERING (AQ >= 6)")
print("="*80)
print("\n⚠️  CRITICAL: Matching C4 preprocessing by filtering ASC cases with AQ < 6")
print("   C4 dataset excluded autism cases with AQ < 6 to create cleaner dataset")
print("   This ensures CARD preprocessing matches C4 exactly\n")

# Check if AQ total exists (should be created in Step 4)
if 'aq_total' not in df_card_aggregated.columns:
    print("⚠️  AQ total not found - calculating from AQ items...")
    aq_cols = [f'aq_{i}' for i in range(1, 11)]
    existing_aq_cols = [col for col in aq_cols if col in df_card_aggregated.columns]
    if existing_aq_cols:
        df_card_aggregated['aq_total'] = df_card_aggregated[existing_aq_cols].sum(axis=1)
        print(f"  ✅ Calculated AQ total from {len(existing_aq_cols)} items")
    else:
        print("  ⚠️  No AQ items found - cannot apply AQ filtering")
        print("  Proceeding without AQ filtering (may reduce performance)")
        df_card_aggregated['aq_total'] = 0  # Placeholder

# Check AQ distribution for ASC cases
if 'autism_target' in df_card_aggregated.columns:
    asc_cases = df_card_aggregated[df_card_aggregated['autism_target'] == 1]
    if len(asc_cases) > 0:
        print(f"\nASC cases AQ distribution:")
        print(f"  Total ASC cases: {len(asc_cases)}")
        if 'aq_total' in asc_cases.columns:
            print(f"  Mean AQ: {asc_cases['aq_total'].mean():.2f}")
            print(f"  Median AQ: {asc_cases['aq_total'].median():.2f}")
            print(f"  Min AQ: {asc_cases['aq_total'].min():.0f}")
            print(f"  Max AQ: {asc_cases['aq_total'].max():.0f}")
            print(f"  Cases with AQ < 6: {(asc_cases['aq_total'] < 6).sum()}")
            print(f"  Cases with AQ >= 6: {(asc_cases['aq_total'] >= 6).sum()}")
            print(f"  Percentage with AQ < 6: {(asc_cases['aq_total'] < 6).sum() / len(asc_cases) * 100:.1f}%")
            
            # Apply filtering: Remove ASC cases with AQ < 6 (matching C4)
            before_aq_filter = len(df_card_aggregated)
            df_card_filtered_aq = df_card_aggregated[
                ~((df_card_aggregated['autism_target'] == 1) & (df_card_aggregated['aq_total'] < 6))
            ].copy()
            after_aq_filter = len(df_card_filtered_aq)
            removed_aq = before_aq_filter - after_aq_filter
            
            print(f"\n✅ Applied AQ threshold filtering:")
            print(f"   Before: {before_aq_filter} participants")
            print(f"   After: {after_aq_filter} participants")
            print(f"   Removed: {removed_aq} ASC cases with AQ < 6")
            
            if removed_aq > 0:
                print(f"\n   This matches C4 preprocessing exactly")
                print(f"   Removed cases may be misdiagnosed or borderline")
                print(f"   This should improve model performance by creating cleaner dataset")
            
            df_card_aggregated = df_card_filtered_aq
        else:
            print("  ⚠️  AQ total not available - skipping AQ filtering")
    else:
        print("\n⚠️  No ASC cases found - skipping AQ filtering")
else:
    print("\n⚠️  autism_target not found - skipping AQ filtering")

# Step 1: Filter out adolescent and child questionnaires
print("\n" + "="*80)
print("STEP 1: FILTERING OUT ADOLESCENT/CHILD QUESTIONNAIRES")
print("="*80)

# Check if we have TestName column to identify adolescent/child questionnaires
test_name_col = None
for col in df_card_aggregated.columns:
    if 'test' in col.lower() and 'name' in col.lower():
        test_name_col = col
        break

if test_name_col:
    print(f"\nFound test name column: '{test_name_col}'")
    
    # Identify rows with adolescent or child questionnaires
    test_names = df_card_aggregated[test_name_col].astype(str).str.lower()
    adolescent_mask = test_names.str.contains('adolescent|child', case=False, na=False)
    
    n_adolescent = adolescent_mask.sum()
    print(f"  Rows with adolescent/child questionnaires: {n_adolescent}")
    
    if n_adolescent > 0:
        # Check which participants have ONLY adolescent/child questionnaires
        # We want to drop participants who ONLY have adolescent/child data
        # But keep participants who have adult questionnaires
        
        # Get participant ID column
        participant_id_col = None
        for col in df_card_aggregated.columns:
            if 'userid' in col.lower() or 'volunteer' in col.lower() or 'participant' in col.lower():
                participant_id_col = col
                break
        
        if participant_id_col:
            # Find participants who have ONLY adolescent/child questionnaires
            participants_with_adolescent = set(df_card_aggregated[adolescent_mask][participant_id_col].unique())
            participants_with_adult = set(df_card_aggregated[~adolescent_mask][participant_id_col].unique())
            participants_only_adolescent = participants_with_adolescent - participants_with_adult
            
            print(f"  Participants with adolescent/child data: {len(participants_with_adolescent)}")
            print(f"  Participants with adult data: {len(participants_with_adult)}")
            print(f"  Participants with ONLY adolescent/child data: {len(participants_only_adolescent)}")
            
            # Drop participants who only have adolescent/child questionnaires
            if len(participants_only_adolescent) > 0:
                df_card_filtered = df_card_aggregated[~df_card_aggregated[participant_id_col].isin(participants_only_adolescent)].copy()
                print(f"\n  ✅ Dropped {len(df_card_aggregated) - len(df_card_filtered)} rows from participants with only adolescent/child data")
                df_card_aggregated = df_card_filtered
            else:
                print(f"\n  ✅ All participants have adult questionnaire data - no participants dropped")
        else:
            # If no participant ID, just drop rows with adolescent questionnaires
            df_card_aggregated = df_card_aggregated[~adolescent_mask].copy()
            print(f"\n  ✅ Dropped {n_adolescent} rows with adolescent/child questionnaires")
    else:
        print("  ✅ No adolescent/child questionnaires found")
else:
    print("\n⚠️  No test name column found - cannot filter adolescent/child questionnaires")
    print("  Proceeding with all data")

# Step 2: Downsample to create 50/50 split
print("\n" + "="*80)
print("STEP 2: DOWNSAMPLING TO CREATE 50/50 SPLIT")
print("="*80)

# Separate cases and controls
cases = df_card_aggregated[df_card_aggregated['autism_target'] == 1].copy()
controls = df_card_aggregated[df_card_aggregated['autism_target'] == 0].copy()

print(f"\nDataset before downsampling:")
print(f"  Cases (autism): {len(cases)}")
print(f"  Controls (non-autism): {len(controls)}")
print(f"  Imbalance ratio: {len(controls)/len(cases):.2f}:1")

# Downsample to create TRUE 50/50 split: match the smaller class
# After AQ filtering, ASC cases may be fewer (e.g. 508), so we balance to that
target_n = min(len(cases), len(controls))  # True 50/50: same number in each class

if len(cases) > target_n:
    cases_downsampled = resample(
        cases,
        replace=False,
        n_samples=target_n,
        random_state=42
    )
    print(f"\n✅ Downsampled cases: {len(cases)} → {len(cases_downsampled)}")
else:
    cases_downsampled = cases.copy()
    print(f"\n✅ Using all available cases: {len(cases_downsampled)}")

if len(controls) > target_n:
    controls_downsampled = resample(
        controls, 
        replace=False, 
        n_samples=target_n, 
        random_state=42
    )
    print(f"✅ Downsampled controls: {len(controls)} → {len(controls_downsampled)}")
else:
    controls_downsampled = controls.copy()
    print(f"✅ Using all available controls: {len(controls_downsampled)}")

# Combine downsampled cases and controls
df_card_balanced = pd.concat([cases_downsampled, controls_downsampled], ignore_index=True)

# Shuffle
df_card_balanced = df_card_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Balanced dataset created")
print(f"   Shape: {df_card_balanced.shape}")
print(f"   Autism target distribution:")
print(df_card_balanced['autism_target'].value_counts())
print(f"   Balance ratio: {(df_card_balanced['autism_target'] == 0).sum()}:{(df_card_balanced['autism_target'] == 1).sum()}")

# Save balanced dataset
balanced_output_path = os.path.join(OUTPUT_DIR, 'card_external_validation_balanced.csv')
df_card_balanced.to_csv(balanced_output_path, index=False)
print(f"\n✅ Balanced dataset saved to: {balanced_output_path}")

# CRITICAL: Replace df_card_aggregated with balanced dataset for subsequent steps (Steps 6-9)
print(f"\n" + "="*80)
print("UPDATING df_card_aggregated TO USE BALANCED DATASET")
print("="*80)
df_card_aggregated = df_card_balanced.copy()
print(f"   Replaced df_card_aggregated: {original_size} → {len(df_card_aggregated)} participants")
print(f"   ✅ Steps 6-9 will now use the balanced 50/50 dataset ({len(cases_downsampled)}:{len(controls_downsampled)})")
print(f"   ✅ This ensures feature alignment, scaling, predictions, and evaluation use balanced data")

print(f"\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"   Original dataset (before any filtering): {original_size} participants")
if 'removed_aq' in locals() and removed_aq > 0:
    print(f"   After AQ filtering (ASC cases with AQ < 6 removed): {before_aq_filter - removed_aq} participants")
print(f"   After filtering adolescent/child: {len(cases) + len(controls)} participants")
print(f"   Cases (autism): {len(cases)} → {len(cases_downsampled)}")
print(f"   Controls (non-autism): {len(controls)} → {len(controls_downsampled)}")
print(f"   Final balanced dataset: {len(df_card_balanced)} participants (50/50 split: {len(cases_downsampled)}:{len(controls_downsampled)})")
print(f"   Output file: {balanced_output_path}")
print(f"\n   ✅ AQ threshold filtering applied (matching C4 preprocessing)")
print(f"   ✅ Adolescent/child data filtered")
print(f"   ✅ Balanced to true 50/50 split ({len(cases_downsampled)} each class)")
print(f"\n   ⚠️  NOTE: df_card_aggregated has been updated to the balanced dataset.")
print(f"      Steps 6-9 will now process the balanced 50/50 dataset ({len(df_card_balanced)} participants)")

## STEP 12: Diagnostic Analysis - Why Performance is Poor

Investigating potential causes of poor model performance on CARD dataset.

In [ ]:
print("="*80)
print("DIAGNOSTIC ANALYSIS: INVESTIGATING POOR PERFORMANCE")
print("="*80)

import numpy as np
from scipy import stats

# Require Step 8 (Generate Predictions) to have been run first
loaded_models = globals().get('loaded_models', {})
probabilities = globals().get('probabilities', {})
predictions = globals().get('predictions', {})
if not loaded_models:
    print("\nRun Step 8 (Generate Predictions) first. This cell needs: loaded_models, probabilities, predictions.")
    print("Execution order: Steps 1-11 -> 6-7 -> 8 -> then Step 12 (this diagnostic).")
    print("Skipping prediction-distribution section; rest of diagnostic may run if X_card etc. exist.")

# 1. Check prediction distributions
print("\n" + "="*80)
print("1. PREDICTION DISTRIBUTIONS")
print("="*80)
for name in loaded_models.keys():
    y_proba = probabilities[name]
    y_pred = predictions[name]
    print(f"\n{name}:")
    print(f"  Probability stats: min={y_proba.min():.4f}, max={y_proba.max():.4f}, mean={y_proba.mean():.4f}, median={np.median(y_proba):.4f}")
    print(f"  Predicted positives: {y_pred.sum()} ({y_pred.mean()*100:.1f}%)")
    print(f"  Predicted negatives: {(y_pred == 0).sum()} ({(y_pred == 0).mean()*100:.1f}%)")
    print(f"  Probability > 0.5: {(y_proba > 0.5).sum()} ({(y_proba > 0.5).mean()*100:.1f}%)")
    print(f"  Probability > 0.3: {(y_proba > 0.3).sum()} ({(y_proba > 0.3).mean()*100:.1f}%)")

# 2. Check feature distributions (compare key features)
print("\n" + "="*80)
print("2. FEATURE DISTRIBUTION ANALYSIS")
print("="*80)

# Load C4 training data for comparison (optional diagnostic)
c4_data_candidates = [os.path.join(OUTPUT_DIR, 'data_c4_processed.csv'), os.path.join(ARTIFACT_DIR, '..', 'data', 'processed', 'data_c4_processed.csv'), os.path.join(ARTIFACT_DIR, '..', 'data_c4_processed.csv'), 'data/processed/data_c4_processed.csv']
c4_data_path = next((p for p in c4_data_candidates if os.path.exists(p)), None)
if c4_data_path and 'X_card' in globals():
    print("\nLoading C4 training data for comparison...")
    df_c4 = pd.read_csv(c4_data_path)
    
    # Compare key features
    key_features = ['spq_total', 'eq_total', 'sqr_total', 'd_score', 'age', 'sex_num']
    available_features = [f for f in key_features if f in X_card.columns and f in df_c4.columns]
    
    print(f"\nComparing {len(available_features)} key features:")
    for feat in available_features:
        c4_vals = df_c4[feat].dropna()
        card_vals = X_card[feat].dropna()
        
        if len(c4_vals) > 0 and len(card_vals) > 0:
            print(f"\n  {feat}:")
            print(f"    C4:   mean={c4_vals.mean():.2f}, std={c4_vals.std():.2f}, range=[{c4_vals.min():.2f}, {c4_vals.max():.2f}]")
            print(f"    CARD: mean={card_vals.mean():.2f}, std={card_vals.std():.2f}, range=[{card_vals.min():.2f}, {card_vals.max():.2f}]")
            
            # Check if distributions are significantly different
            try:
                stat, pval = stats.ks_2samp(c4_vals, card_vals)
                print(f"    KS test: stat={stat:.4f}, p={pval:.4e} {'***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''}")
            except:
                pass
else:
    if not c4_data_path:
        print("\n⚠️  C4 training data not found for comparison")
    else:
        print("\n⚠️  X_card not available (run Step 7 first) - skipping C4 comparison")

# 3. Check for zero-filled or missing features
print("\n" + "="*80)
print("3. MISSING/ZERO-FILLED FEATURES")
print("="*80)
if 'X_card' not in globals():
    print("\n⚠️  X_card not available (run Step 7 first) - skipping sections 3-4")
else:
    zero_features = []
    low_variance_features = []
    for feat in X_card.columns:
        if X_card[feat].nunique() == 1:
            zero_features.append(feat)
        elif X_card[feat].std() < 0.01:
            low_variance_features.append(feat)

    if zero_features:
        print(f"\n⚠️  Found {len(zero_features)} features with zero variance (constant values):")
        for feat in zero_features[:10]:
            print(f"    - {feat}: value={X_card[feat].iloc[0]}")
        if len(zero_features) > 10:
            print(f"    ... and {len(zero_features) - 10} more")

    if low_variance_features:
        print(f"\n⚠️  Found {len(low_variance_features)} features with very low variance (< 0.01):")
        for feat in low_variance_features[:10]:
            print(f"    - {feat}: std={X_card[feat].std():.6f}")
        if len(low_variance_features) > 10:
            print(f"    ... and {len(low_variance_features) - 10} more")

    # 4. Check scaled feature statistics
    if 'X_card_scaled' in globals():
        print("\n" + "="*80)
        print("4. SCALED FEATURE STATISTICS")
        print("="*80)
        print(f"\nScaled feature matrix shape: {X_card_scaled.shape}")
        print(f"Scaled feature stats:")
        print(f"  Mean: {X_card_scaled.mean(axis=0).mean():.4f} (should be ~0)")
        print(f"  Std:  {X_card_scaled.std(axis=0).mean():.4f} (should be ~1)")
        print(f"  Min:  {X_card_scaled.min(axis=0).min():.4f}")
        print(f"  Max:  {X_card_scaled.max(axis=0).max():.4f}")
        extreme_mask = (np.abs(X_card_scaled) > 5).any(axis=1)
        print(f"\n⚠️  Samples with extreme scaled values (|z| > 5): {extreme_mask.sum()} ({extreme_mask.mean()*100:.1f}%)")

# 5. Check questionnaire totals
print("\n" + "="*80)
print("5. QUESTIONNAIRE TOTALS CHECK")
print("="*80)
questionnaire_totals = ['spq_total', 'eq_total', 'sqr_total']
for q_total in questionnaire_totals:
    if q_total in df_card_aggregated.columns:
        vals = df_card_aggregated[q_total]
        print(f"\n  {q_total}:")
        print(f"    Non-zero: {(vals != 0).sum()} ({(vals != 0).mean()*100:.1f}%)")
        print(f"    Mean: {vals.mean():.2f}, Median: {vals.median():.2f}")
        print(f"    Range: [{vals.min():.0f}, {vals.max():.0f}]")
        if (vals == 0).sum() > len(vals) * 0.5:
            print(f"    ⚠️  WARNING: {(vals == 0).sum()} samples ({((vals == 0).mean()*100):.1f}%) have zero total!")

# 6. Summary and recommendations
print("\n" + "="*80)
print("6. SUMMARY AND POTENTIAL ISSUES")
print("="*80)
print("\nPotential causes of poor performance:")
print("  1. Domain shift: CARD data may have different distributions than C4 training data")
print("  2. Feature misalignment: Some features may be missing or incorrectly calculated")
print("  3. Data quality: Missing questionnaire data or incorrect scoring")
print("  4. Scaling issues: Features may not be properly scaled or have extreme values")
print("  5. Model calibration: Models may need threshold tuning for CARD dataset")
print("\nRecommendations:")
print("  - Check if questionnaire items were correctly extracted and scored")
print("  - Verify feature engineering matches C4 exactly")
print("  - Consider domain adaptation techniques")
print("  - Try threshold tuning (models may be too conservative)")
print("  - Check if CARD data quality/preprocessing differs from C4")

## STEP 13: Model Performance Improvements & Diagnostics

**IMPORTANT METHODOLOGICAL NOTE:**

This section includes diagnostic and exploratory analysis techniques to understand model performance and identify potential improvements.

**For proper external validation**, we report performance using models exactly as trained on C4 (default 0.5 threshold). The final validation metrics come from Step 9.

The techniques included here (RobustScaler, CORAL alignment, feature verification) are:
- **Diagnostic tools** to understand why performance is poor
- **Exploratory analysis** to identify potential improvements
- **NOT** as validation metrics (those remain from Step 9)

In [ ]:
# THRESHOLD OPTIMIZATION REMOVED
# Threshold optimization on validation data constitutes data leakage and is not scientifically robust.
# All validation metrics use the default 0.5 threshold (matching C4 evaluation methodology).
# See Step 9 for validation results.

In [ ]:
print("="*80)
print("IMPROVEMENT 2: ROBUST SCALING (HANDLE EXTREME VALUES)")
print("="*80)
print("\nUsing RobustScaler instead of StandardScaler to handle outliers and domain shift...")
print("⚠️  NOTE: This refits the scaler, which is acceptable for preprocessing,")
print("   but we still use default 0.5 threshold for validation metrics.")

from sklearn.preprocessing import RobustScaler

# Load C4 training data to fit RobustScaler (optional - for diagnostic comparison only)
# Try multiple possible locations for C4 processed data
c4_data_candidates = [
    os.path.join(OUTPUT_DIR, 'data_c4_processed.csv'),
    os.path.join(ARTIFACT_DIR, '..', 'data', 'processed', 'data_c4_processed.csv'),
    os.path.join(ARTIFACT_DIR, '..', 'data_c4_processed.csv'),
    'data/processed/data_c4_processed.csv',
]
c4_data_path = None
for p in c4_data_candidates:
    if os.path.exists(p):
        c4_data_path = p
        break
if c4_data_path and os.path.exists(c4_data_path):
    print("Loading C4 training data...")
    df_c4_train = pd.read_csv(c4_data_path)
    
    # Build C4 feature matrix (same as CARD)
    X_c4_train = pd.DataFrame(index=df_c4_train.index)
    for feat_name in c4_feature_names:
        if feat_name in df_c4_train.columns:
            X_c4_train[feat_name] = df_c4_train[feat_name]
        else:
            X_c4_train[feat_name] = 0
    X_c4_train = X_c4_train[c4_feature_names]
    X_c4_train = X_c4_train.fillna(0)
    
    # Fit RobustScaler on C4 training data
    robust_scaler = RobustScaler()
    X_c4_robust_scaled = robust_scaler.fit_transform(X_c4_train)
    
    # Transform CARD data with RobustScaler
    X_card_robust_scaled = robust_scaler.transform(X_card)
    
    print(f"✅ RobustScaler fitted on C4 training data ({X_c4_train.shape[0]} samples)")
    print(f"✅ CARD data transformed: {X_card_robust_scaled.shape}")
    print(f"\nRobustScaler statistics:")
    print(f"  CARD scaled mean: {X_card_robust_scaled.mean(axis=0).mean():.4f}")
    print(f"  CARD scaled std: {X_card_robust_scaled.std(axis=0).mean():.4f}")
    print(f"  Extreme values (|z| > 5): {(np.abs(X_card_robust_scaled) > 5).any(axis=1).sum()} samples")
    
    # Test models with RobustScaler
    print("\n" + "="*80)
    print("TESTING MODELS WITH ROBUST SCALER")
    print("="*80)
    
    robust_predictions = {}
    robust_probabilities = {}
    robust_results = {}
    
    for name, model in loaded_models.items():
        y_proba = model.predict_proba(X_card_robust_scaled)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)
        
        robust_probabilities[name] = y_proba
        robust_predictions[name] = y_pred
        
        # Use default 0.5 threshold for validation comparison
        robust_results[name] = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_proba)
        }
        
        print(f"\n{name} (RobustScaler, default 0.5 threshold):")
        print(f"  F1: {robust_results[name]['f1']:.4f}")
        print(f"  Recall: {robust_results[name]['recall']:.4f}")
        print(f"  AUC: {robust_results[name]['auc']:.4f}")
    
    # Compare RobustScaler vs StandardScaler (both with default 0.5 threshold)
    print("\n" + "="*80)
    print("ROBUST SCALER vs STANDARD SCALER COMPARISON (Default 0.5 Threshold)")
    print("="*80)
    robust_comparison = []
    for name in loaded_models.keys():
        # Get StandardScaler results with default threshold
        standard_f1 = f1_score(y_true, predictions[name], zero_division=0)
        standard_recall = recall_score(y_true, predictions[name], zero_division=0)
        
        robust_comparison.append({
            'Model': name,
            'StandardScaler_F1': standard_f1,
            'RobustScaler_F1': robust_results[name]['f1'],
            'StandardScaler_Recall': standard_recall,
            'RobustScaler_Recall': robust_results[name]['recall'],
            'F1_Change': robust_results[name]['f1'] - standard_f1,
            'Recall_Change': robust_results[name]['recall'] - standard_recall
        })
    
    robust_comparison_df = pd.DataFrame(robust_comparison)
    print(robust_comparison_df.round(4).sort_values('F1_Change', ascending=False))
    
else:
    print("⚠️  C4 training data file not found - skipping RobustScaler diagnostic")
    print("   (Models are still loaded from ARTIFACT_DIR and are correct.)")
    print("   C4 data is only needed for optional diagnostics; validation uses StandardScaler.)")
    print("   Using StandardScaler results only")

In [ ]:
print("="*80)
print("IMPROVEMENT 3: FEATURE VERIFICATION & DATA QUALITY CHECK")
print("="*80)

# Check questionnaire totals
print("\n1. QUESTIONNAIRE TOTALS VERIFICATION:")
questionnaire_features = {
    'SPQ': ['spq_total'] + [f'spq_{i}' for i in range(1, 11)],
    'EQ': ['eq_total'] + [f'eq_{i}' for i in range(1, 11)],
    'SQR': ['sqr_total'] + [f'sqr_{i}' for i in range(1, 11)]
}

for q_name, q_features in questionnaire_features.items():
    total_col = f'{q_name.lower()}_total'
    item_cols = [f'{q_name.lower()}_{i}' for i in range(1, 11)]
    
    if total_col in df_card_aggregated.columns:
        # Check if totals match sum of items
        available_items = [col for col in item_cols if col in df_card_aggregated.columns]
        if available_items:
            calculated_total = df_card_aggregated[available_items].sum(axis=1)
            actual_total = df_card_aggregated[total_col]
            
            mismatches = (np.abs(calculated_total - actual_total) > 0.01).sum()
            zero_totals = (actual_total == 0).sum()
            
            print(f"\n  {q_name}:")
            print(f"    Items available: {len(available_items)}/10")
            print(f"    Total mismatches: {mismatches} ({mismatches/len(df_card_aggregated)*100:.1f}%)")
            print(f"    Zero totals: {zero_totals} ({zero_totals/len(df_card_aggregated)*100:.1f}%)")
            print(f"    Mean total: {actual_total.mean():.2f}, Range: [{actual_total.min():.0f}, {actual_total.max():.0f}]")
            
            if mismatches > len(df_card_aggregated) * 0.1:
                print(f"    ⚠️  WARNING: High mismatch rate - may indicate scoring issues")
            if zero_totals > len(df_card_aggregated) * 0.5:
                print(f"    ⚠️  WARNING: High zero rate - may indicate missing data")

# Check feature distributions
print("\n2. FEATURE DISTRIBUTION CHECKS:")
print("\n  Checking for features with zero variance or extreme distributions...")
zero_var_features = []
low_var_features = []
extreme_features = []

for feat in X_card.columns:
    vals = X_card[feat].dropna()
    if len(vals) == 0:
        continue
    
    if vals.nunique() == 1:
        zero_var_features.append(feat)
    elif vals.std() < 0.01:
        low_var_features.append(feat)
    
    # Check for extreme values (more than 5 standard deviations from mean)
    z_scores = np.abs((vals - vals.mean()) / (vals.std() + 1e-8))
    if (z_scores > 5).sum() > len(vals) * 0.05:  # More than 5% extreme
        extreme_features.append(feat)

if zero_var_features:
    print(f"\n  ⚠️  Zero variance features ({len(zero_var_features)}): {zero_var_features[:5]}")
if low_var_features:
    print(f"\n  ⚠️  Low variance features ({len(low_var_features)}): {low_var_features[:5]}")
if extreme_features:
    print(f"\n  ⚠️  Features with extreme values ({len(extreme_features)}): {extreme_features[:5]}")

# Check missing data patterns
print("\n3. MISSING DATA PATTERNS:")

# Ensure X_card and df_card_aggregated have matching indices
# After Step 11 filtering, indices may not match, so we need to align them
if X_card.shape[0] != len(df_card_aggregated):
    print(f"\n  ⚠️  Size mismatch detected:")
    print(f"     X_card has {X_card.shape[0]} samples")
    print(f"     df_card_aggregated has {len(df_card_aggregated)} samples")
    print(f"     This likely means Step 11 (AQ filtering) was run after Step 7")
    print(f"\n  ✅ SOLUTION: Re-run Steps 6-7 after Step 11 to regenerate features")
    print(f"     OR: Use X_card.loc[df_card_aggregated.index] if indices align")
    
    # Try to align by index if possible
    if X_card.index.isin(df_card_aggregated.index).any():
        print(f"\n  Attempting to align by index...")
        try:
            # Filter X_card to only include rows that exist in df_card_aggregated
            X_card = X_card.loc[X_card.index.isin(df_card_aggregated.index)]
            # Reindex to match df_card_aggregated
            X_card = X_card.reindex(df_card_aggregated.index)
            print(f"  ✅ Successfully aligned X_card to df_card_aggregated index")
        except Exception as e:
            print(f"  ❌ Failed to align: {e}")
            print(f"  Please re-run Steps 6-7 after Step 11")
            raise ValueError("X_card and df_card_aggregated indices don't align. Please re-run Steps 6-7 after Step 11.")
    else:
        print(f"  ❌ Indices don't overlap - cannot align automatically")
        print(f"  Please re-run Steps 6-7 after Step 11")
        raise ValueError("X_card and df_card_aggregated have incompatible indices. Please re-run Steps 6-7 after Step 11.")

missing_by_class = {}
for target_val in [0, 1]:
    mask = df_card_aggregated['autism_target'] == target_val
    # Use .loc to ensure index alignment
    subset = X_card.loc[mask]
    missing_counts = (subset == 0).sum(axis=0)
    missing_by_class[target_val] = missing_counts

print(f"\n  Features with >50% zeros in autism cases: {(missing_by_class[1] > len(X_card.loc[df_card_aggregated['autism_target']==1]) * 0.5).sum()}")
print(f"  Features with >50% zeros in non-autism cases: {(missing_by_class[0] > len(X_card.loc[df_card_aggregated['autism_target']==0]) * 0.5).sum()}")

# Check key demographic features
print("\n4. DEMOGRAPHIC FEATURES CHECK:")
demo_features = ['age', 'sex_num', 'age_group_19-30', 'age_group_31-45', 'age_group_46-60', 'age_group_61+']
for feat in demo_features:
    if feat in df_card_aggregated.columns:
        vals = df_card_aggregated[feat]
        print(f"  {feat}: mean={vals.mean():.2f}, unique={vals.nunique()}, missing={(vals==0).sum() if feat != 'age' else vals.isna().sum()}")

print("\n✅ Feature verification complete")

In [ ]:
print("="*80)
print("IMPROVEMENT 4: DOMAIN ADAPTATION - CORAL ALIGNMENT")
print("="*80)
print("\nUsing CORAL (CORrelation ALignment) to reduce domain shift between C4 and CARD...")

try:
    from sklearn.decomposition import PCA
    from scipy.linalg import sqrtm
    
    def coral_alignment(X_s, X_t):
        """
        CORAL: Correlation Alignment for Domain Adaptation
        X_s: source domain (C4 training)
        X_t: target domain (CARD)
        """
        # Center the data
        X_s_mean = X_s.mean(axis=0)
        X_t_mean = X_t.mean(axis=0)
        X_s_centered = X_s - X_s_mean
        X_t_centered = X_t - X_t_mean
        
        # Compute covariance matrices
        cov_s = np.cov(X_s_centered.T)
        cov_t = np.cov(X_t_centered.T)
        
        # Compute whitening and coloring transformations
        try:
            # Whitening: X_s -> white space
            cov_s_sqrt = sqrtm(cov_s + np.eye(cov_s.shape[0]) * 1e-5)
            # Coloring: white space -> X_t space
            cov_t_sqrt = sqrtm(cov_t + np.eye(cov_t.shape[0]) * 1e-5)
            
            # Transformation matrix
            A = cov_t_sqrt @ np.linalg.inv(cov_s_sqrt)
            
            # Transform source data
            X_s_aligned = X_s_centered @ A.T + X_t_mean
            
            return X_s_aligned, A
        except:
            # Fallback: use PCA if matrix inversion fails
            print("    ⚠️  Matrix inversion failed, using PCA alignment instead")
            pca = PCA(n_components=min(20, X_s.shape[1]))
            X_s_pca = pca.fit_transform(X_s_centered)
            X_t_pca = pca.transform(X_t_centered)
            return X_t_pca, None
    
    # Apply CORAL alignment
    if 'X_c4_train' in locals() and X_c4_train.shape[0] > 0:
        print("Applying CORAL alignment...")
        
        # Use RobustScaler-scaled data if available
        if 'X_card_robust_scaled' in locals():
            X_card_for_coral = X_card_robust_scaled
            X_c4_for_coral = X_c4_robust_scaled
        else:
            X_card_for_coral = X_card_scaled
            X_c4_for_coral = X_c4_scaled
        
        # Align CARD to C4 distribution
        X_card_coral_aligned, coral_transform = coral_alignment(X_c4_for_coral, X_card_for_coral)
        
        print(f"✅ CORAL alignment complete")
        print(f"   CARD aligned shape: {X_card_coral_aligned.shape}")
        
        # Test models with CORAL-aligned data
        print("\n" + "="*80)
        print("TESTING MODELS WITH CORAL-ALIGNED DATA")
        print("="*80)
        
        coral_predictions = {}
        coral_probabilities = {}
        coral_results = {}
        
        for name, model in loaded_models.items():
            y_proba = model.predict_proba(X_card_coral_aligned)[:, 1]
            
            # Use default 0.5 threshold for validation
            y_pred = (y_proba >= 0.5).astype(int)
            
            coral_probabilities[name] = y_proba
            coral_predictions[name] = y_pred
            
            coral_results[name] = {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision': precision_score(y_true, y_pred, zero_division=0),
                'recall': recall_score(y_true, y_pred, zero_division=0),
                'f1': f1_score(y_true, y_pred, zero_division=0),
                'auc': roc_auc_score(y_true, y_proba)
            }
            
            print(f"\n{name} (CORAL, default 0.5 threshold):")
            print(f"  F1: {coral_results[name]['f1']:.4f}")
            print(f"  Recall: {coral_results[name]['recall']:.4f}")
            print(f"  AUC: {coral_results[name]['auc']:.4f}")
        
        # Compare CORAL vs baseline
        print("\n" + "="*80)
        print("CORAL ALIGNMENT vs BASELINE COMPARISON")
        print("="*80)
        coral_comparison = []
        for name in loaded_models.keys():
            coral_comparison.append({
                'Model': name,
                'Baseline_F1': optimized_metrics[name]['f1'],
                'CORAL_F1': coral_results[name]['f1'],
                'Baseline_Recall': optimized_metrics[name]['recall'],
                'CORAL_Recall': coral_results[name]['recall'],
                'F1_Change': coral_results[name]['f1'] - optimized_metrics[name]['f1'],
                'Recall_Change': coral_results[name]['recall'] - optimized_metrics[name]['recall']
            })
        
        coral_comparison_df = pd.DataFrame(coral_comparison)
        print(coral_comparison_df.round(4).sort_values('F1_Change', ascending=False))
        
    else:
        print("⚠️  C4 training data not available - skipping CORAL alignment")
        
except ImportError:
    print("⚠️  scipy not available - skipping CORAL alignment")
except Exception as e:
    print(f"⚠️  CORAL alignment failed: {e}")
    print("   Continuing with other improvements...")

In [ ]:
print("="*80)
print("DIAGNOSTIC 5: ENSEMBLE METHODS (EXPLORATORY)")
print("="*80)
print("\n⚠️  IMPORTANT: Ensemble methods combine multiple approaches.")
print("   For validation, use individual model results from Step 9.")
print("   Ensembles are shown here as exploratory analysis only.")
print("\nCreating ensemble predictions from multiple models and scaling methods...")

# Check if y_true matches current dataset size
y_true = df_card_aggregated['autism_target'].values
current_n_samples = len(y_true)

# Verify that probabilities match current dataset size
if 'probabilities' in locals() and len(probabilities) > 0:
    first_proba = list(probabilities.values())[0]
    if len(first_proba) != current_n_samples:
        print(f"\n⚠️  Size mismatch detected in ensemble cell:")
        print(f"   Current dataset: {current_n_samples} samples")
        print(f"   Existing probabilities: {len(first_proba)} samples")
        print(f"   Please run Step 8 again after Step 11 (AQ filtering) to regenerate predictions")
        raise ValueError(f"Size mismatch: probabilities have {len(first_proba)} samples but dataset has {current_n_samples} samples. Please regenerate predictions.")

# Collect all prediction methods
all_probabilities = {}
all_methods = []

# Baseline (StandardScaler with default 0.5 threshold)
all_methods.append('Baseline_StandardScaler')
all_probabilities['Baseline_StandardScaler'] = probabilities

# RobustScaler
if 'robust_probabilities' in locals():
    all_methods.append('RobustScaler')
    all_probabilities['RobustScaler'] = robust_probabilities

# CORAL
if 'coral_probabilities' in locals():
    all_methods.append('CORAL')
    all_probabilities['CORAL'] = coral_probabilities

print(f"\nAvailable methods: {all_methods}")

# Create ensemble predictions
ensemble_results = {}

# Method 1: Simple average of probabilities
if len(all_methods) > 1:
    print("\n1. SIMPLE AVERAGE ENSEMBLE:")
    avg_proba = np.mean([all_probabilities[method] for method in all_methods], axis=0)
    
    # Optimize threshold for ensemble
    precisions, recalls, thresholds = precision_recall_curve(y_true, avg_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores[:-1])
    best_threshold = thresholds[best_idx]
    
    ensemble_pred = (avg_proba >= best_threshold).astype(int)
    
    ensemble_results['Simple_Average'] = {
        'threshold': best_threshold,
        'accuracy': accuracy_score(y_true, ensemble_pred),
        'precision': precision_score(y_true, ensemble_pred, zero_division=0),
        'recall': recall_score(y_true, ensemble_pred, zero_division=0),
        'f1': f1_score(y_true, ensemble_pred, zero_division=0),
        'auc': roc_auc_score(y_true, avg_proba)
    }
    
    print(f"  Optimal threshold: {best_threshold:.4f}")
    print(f"  F1: {ensemble_results['Simple_Average']['f1']:.4f}")
    print(f"  Recall: {ensemble_results['Simple_Average']['recall']:.4f}")
    print(f"  AUC: {ensemble_results['Simple_Average']['auc']:.4f}")

# Method 2: Weighted average (weight by individual F1 scores)
if len(all_methods) > 1:
    print("\n2. WEIGHTED AVERAGE ENSEMBLE (by F1 score):")
    
    # Calculate weights based on individual F1 scores
    method_weights = {}
    for method in all_methods:
        # Get best F1 for this method
        best_f1 = 0
        for name in loaded_models.keys():
            if method == 'Baseline_StandardScaler':
                f1 = optimized_metrics[name]['f1']
            elif method == 'RobustScaler' and 'robust_results' in locals():
                f1 = robust_results[name]['f1']
            elif method == 'CORAL' and 'coral_results' in locals():
                f1 = coral_results[name]['f1']
            else:
                f1 = 0
            best_f1 = max(best_f1, f1)
        method_weights[method] = best_f1
    
    # Normalize weights
    total_weight = sum(method_weights.values())
    if total_weight > 0:
        method_weights = {k: v/total_weight for k, v in method_weights.items()}
        
        weighted_proba = np.zeros(len(y_true))
        for method in all_methods:
            # Average probabilities across models for this method
            method_avg = np.mean([all_probabilities[method][name] for name in loaded_models.keys()], axis=0)
            weighted_proba += method_weights[method] * method_avg
        
        # Optimize threshold
        precisions, recalls, thresholds = precision_recall_curve(y_true, weighted_proba)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
        best_idx = np.argmax(f1_scores[:-1])
        best_threshold = thresholds[best_idx]
        
        ensemble_pred = (weighted_proba >= best_threshold).astype(int)
        
        ensemble_results['Weighted_Average'] = {
            'threshold': best_threshold,
            'accuracy': accuracy_score(y_true, ensemble_pred),
            'precision': precision_score(y_true, ensemble_pred, zero_division=0),
            'recall': recall_score(y_true, ensemble_pred, zero_division=0),
            'f1': f1_score(y_true, ensemble_pred, zero_division=0),
            'auc': roc_auc_score(y_true, weighted_proba),
            'weights': method_weights
        }
        
        print(f"  Weights: {method_weights}")
        print(f"  Optimal threshold: {best_threshold:.4f}")
        print(f"  F1: {ensemble_results['Weighted_Average']['f1']:.4f}")
        print(f"  Recall: {ensemble_results['Weighted_Average']['recall']:.4f}")
        print(f"  AUC: {ensemble_results['Weighted_Average']['auc']:.4f}")

# Summary comparison
print("\n" + "="*80)
print("FINAL COMPARISON: ALL IMPROVEMENT METHODS")
print("="*80)

# Collect best result from each method
final_comparison = []

# Baseline
best_baseline_f1 = max([optimized_metrics[name]['f1'] for name in loaded_models.keys()])
best_baseline_model = [name for name in loaded_models.keys() if optimized_metrics[name]['f1'] == best_baseline_f1][0]
final_comparison.append({
    'Method': 'Baseline (StandardScaler + Threshold)',
    'Model': best_baseline_model,
    'F1': optimized_metrics[best_baseline_model]['f1'],
    'Recall': optimized_metrics[best_baseline_model]['recall'],
    'AUC': optimized_metrics[best_baseline_model]['auc']
})

# RobustScaler
if 'robust_results' in locals():
    best_robust_f1 = max([robust_results[name]['f1'] for name in loaded_models.keys()])
    best_robust_model = [name for name in loaded_models.keys() if robust_results[name]['f1'] == best_robust_f1][0]
    final_comparison.append({
        'Method': 'RobustScaler (default threshold)',
        'Model': best_robust_model,
        'F1': robust_results[best_robust_model]['f1'],
        'Recall': robust_results[best_robust_model]['recall'],
        'AUC': robust_results[best_robust_model]['auc']
    })

# CORAL
if 'coral_results' in locals():
    best_coral_f1 = max([coral_results[name]['f1'] for name in loaded_models.keys()])
    best_coral_model = [name for name in loaded_models.keys() if coral_results[name]['f1'] == best_coral_f1][0]
    final_comparison.append({
        'Method': 'CORAL (default threshold)',
        'Model': best_coral_model,
        'F1': coral_results[best_coral_model]['f1'],
        'Recall': coral_results[best_coral_model]['recall'],
        'AUC': coral_results[best_coral_model]['auc']
    })

# Ensembles
for ensemble_name, ensemble_metrics in ensemble_results.items():
    final_comparison.append({
        'Method': f'Ensemble ({ensemble_name})',
        'Model': 'Multiple',
        'F1': ensemble_metrics['f1'],
        'Recall': ensemble_metrics['recall'],
        'AUC': ensemble_metrics['auc']
    })

final_comparison_df = pd.DataFrame(final_comparison)
print(final_comparison_df.round(4).sort_values('F1', ascending=False))

# Find best overall method
best_method = final_comparison_df.loc[final_comparison_df['F1'].idxmax()]
print(f"\n✅ BEST METHOD: {best_method['Method']}")
print(f"   F1: {best_method['F1']:.4f}")
print(f"   Recall: {best_method['Recall']:.4f}")
print(f"   AUC: {best_method['AUC']:.4f}")

# Save final comparison
final_comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'card_improvement_comparison.csv'), index=False)
print(f"\n✅ Comparison saved to: {OUTPUT_DIR}/card_improvement_comparison.csv")

## STEP 14: Summary - Validation vs. Exploratory Results

**Clear separation between scientifically robust validation and exploratory diagnostics.**

In [ ]:
print("="*80)
print("VALIDATION vs. EXPLORATORY RESULTS SUMMARY")
print("="*80)

print("\n" + "="*80)
print("SCIENTIFICALLY ROBUST VALIDATION METRICS (Step 9)")
print("="*80)
print("These are the ONLY metrics that should be reported for external validation:")
print("  - Models used exactly as trained on C4")
print("  - Default 0.5 threshold (matching C4 evaluation methodology)")
print("    * C4 results were generated using model.predict() → 0.5 threshold")
print("    * sklearn models use 0.5 threshold by default")
print("    * This ensures fair comparison with C4 training results")
print("  - StandardScaler fitted on C4 training data (not refit on CARD)")
print("\nBest performing model for VALIDATION:")
best_val_model = max(loaded_models.keys(), key=lambda x: roc_auc_score(y_true, probabilities[x]))
best_val_f1 = f1_score(y_true, predictions[best_val_model], zero_division=0)
best_val_recall = recall_score(y_true, predictions[best_val_model], zero_division=0)
best_val_auc = roc_auc_score(y_true, probabilities[best_val_model])

print(f"\n  Model: {best_val_model}")
print(f"  F1: {best_val_f1:.4f}")
print(f"  Recall: {best_val_recall:.4f}")
print(f"  AUC: {best_val_auc:.4f}")
print(f"  Precision: {precision_score(y_true, predictions[best_val_model], zero_division=0):.4f}")

print("\n" + "="*80)
print("EXPLORATORY/DIAGNOSTIC RESULTS (Steps 13)")
print("="*80)
print("These help understand WHY performance is poor and what COULD be improved:")
print("  - Threshold optimization: Shows models are miscalibrated for CARD")
print("  - RobustScaler: Tests if outliers are the issue")
print("  - CORAL: Tests if domain shift can be reduced")
print("  - Ensembles: Shows best-case scenario")
print("\n⚠️  These should NOT be reported as validation metrics")
print("   They indicate potential improvements but require:")
print("   1. Separate validation set for threshold tuning")
print("   2. Retraining models with domain adaptation")
print("   3. Proper cross-validation")

if 'optimized_metrics' in locals():
    print("\nBest exploratory result (threshold optimized):")
    best_exploratory = max(optimized_metrics.keys(), key=lambda x: optimized_metrics[x]['f1'])
    print(f"  Model: {best_exploratory}")
    print(f"  F1: {optimized_metrics[best_exploratory]['f1']:.4f} (vs validation: {best_val_f1:.4f})")
    print(f"  Recall: {optimized_metrics[best_exploratory]['recall']:.4f} (vs validation: {best_val_recall:.4f})")
    print(f"  Improvement: F1 +{optimized_metrics[best_exploratory]['f1'] - best_val_f1:.4f}, Recall +{optimized_metrics[best_exploratory]['recall'] - best_val_recall:.4f}")
    print(f"\n  This shows models COULD achieve better performance IF:")
    print(f"    - Thresholds were tuned on a separate validation set")
    print(f"    - Models were recalibrated for CARD domain")
    print(f"    - But this requires proper methodology (not done here)")

print("\n" + "="*80)
print("RECOMMENDATIONS FOR PROPER VALIDATION")
print("="*80)
print("1. Report Step 9 results as official validation metrics")
print("2. Use exploratory results to understand issues:")
print("   - Low recall suggests model calibration problems")
print("   - Domain shift likely (CARD differs from C4)")
print("   - Consider domain adaptation techniques for future work")
print("3. For improved performance, would need:")
print("   - Separate validation set for threshold tuning")
print("   - Or retrain models with domain adaptation on C4+CARD")
print("   - Proper cross-validation to avoid overfitting")
print("\n✅ Current validation results (Step 9) are scientifically robust")
print("⚠️  Exploratory results (Step 13) are for understanding only")

## STEP 15: Improving Model Generalizability - Training Strategies

**How to optimize C4 model training to make it more generalizable to CARD dataset.**

This section provides strategies for retraining models to improve cross-dataset performance.

In [ ]:
print("="*80)
print("STRATEGIES TO IMPROVE MODEL GENERALIZABILITY")
print("="*80)

print("""
To make C4 models more generalizable to CARD dataset, consider these approaches:

1. DOMAIN ADAPTATION TECHNIQUES:
   a) CORAL (CORrelation ALignment) - Align feature distributions
   b) DANN (Domain Adversarial Neural Networks) - Learn domain-invariant features
   c) Feature alignment - Match CARD feature distributions to C4

2. REGULARIZATION & ROBUSTNESS:
   a) Increase regularization (L1/L2) to prevent overfitting to C4
   b) Use dropout or early stopping
   c) Reduce model complexity

3. DATA AUGMENTATION:
   a) Combine C4 + CARD training data (if ethically appropriate)
   b) Use stratified sampling across datasets
   c) Balance class distributions across datasets

4. FEATURE ENGINEERING:
   a) Focus on domain-invariant features
   b) Remove dataset-specific features
   c) Normalize features consistently across datasets

5. CROSS-DATASET VALIDATION:
   a) Train on C4, validate on CARD (holdout)
   b) Use nested cross-validation across datasets
   c) Monitor performance on both datasets during training

6. ENSEMBLE METHODS:
   a) Train multiple models with different random seeds
   b) Combine predictions from C4-trained and CARD-adapted models
   c) Use stacking with dataset-specific models

7. TRANSFER LEARNING:
   a) Pre-train on C4, fine-tune on CARD subset
   b) Use domain adaptation layers
   c) Progressive training (C4 → CARD)

RECOMMENDED APPROACH FOR YOUR CASE:
Given the large performance drop, domain shift is significant. Consider:

1. IMMEDIATE (No retraining needed):
   - Use RobustScaler instead of StandardScaler (handles outliers better)
   - Apply CORAL alignment (already implemented in Step 13)
   - Use ensemble of multiple scaling methods

2. SHORT-TERM (Retrain with domain adaptation):
   - Train models with increased regularization
   - Use CORAL-aligned features during training
   - Apply early stopping based on CARD validation performance

3. LONG-TERM (Full retraining):
   - Combine C4 + CARD training data (if appropriate)
   - Use domain adaptation techniques (DANN, CORAL)
   - Implement cross-dataset validation
   - Train separate models for each dataset and ensemble

CURRENT STATUS:
- Models trained on C4 only (no domain adaptation)
- Using StandardScaler fitted on C4
- Default 0.5 threshold (matching C4 evaluation)
- Performance drop suggests significant domain shift

NEXT STEPS:
1. Try RobustScaler + CORAL (already implemented in Step 13)
2. If still poor, consider retraining with domain adaptation
3. Evaluate whether CARD data quality/preprocessing differs significantly
""")

print("\n" + "="*80)
print("VERIFICATION: C4 MODEL EVALUATION METHODOLOGY")
print("="*80)

# Verify how C4 models were evaluated
print("\nChecking C4 model evaluation code...")
print("From data_pipeline_recreation.ipynb:")
print("  Code: y_pred = model.predict(X_test_scaled)")
print("  sklearn's model.predict() uses 0.5 threshold by default")
print("  Models are sklearn models (no custom threshold stored)")
print("\n✅ Confirmed: C4 results used default 0.5 threshold")
print("   This matches our CARD validation methodology")

## STEP 16: Practical Implementation - Retraining with Domain Adaptation

**Example code for retraining models with improved generalizability.**

In [ ]:
print("="*80)
print("EXAMPLE: RETRAINING WITH DOMAIN ADAPTATION")
print("="*80)
print("\nThis is a template for retraining models with improved generalizability.")
print("⚠️  NOTE: This requires C4 training data and should be run separately.")
print("   It's provided here as a reference implementation.\n")

retraining_template = '''
# ============================================================================
# TEMPLATE: Retrain Models with Domain Adaptation for Better Generalizability
# ============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_curve
import joblib
import os

# ============================================================================
# STEP 1: Load and Prepare Data
# ============================================================================

# Load C4 training data
df_c4 = pd.read_csv('data/processed/data_c4_processed.csv')
X_c4 = df_c4[c4_feature_names]  # Use same 45 features
y_c4 = df_c4['autism_target']

# Load CARD data (already preprocessed)
# X_card and y_card from current notebook

# ============================================================================
# STEP 2: Apply CORAL Alignment (Domain Adaptation)
# ============================================================================

from scipy.linalg import sqrtm

def coral_alignment(X_s, X_t):
    """CORAL: Correlation Alignment for Domain Adaptation"""
    # Center the data
    X_s_mean = X_s.mean(axis=0)
    X_t_mean = X_t.mean(axis=0)
    X_s_centered = X_s - X_s_mean
    X_t_centered = X_t - X_t_mean
    
    # Compute covariance matrices
    cov_s = np.cov(X_s_centered.T)
    cov_t = np.cov(X_t_centered.T)
    
    # Compute transformation
    cov_s_sqrt = sqrtm(cov_s + np.eye(cov_s.shape[0]) * 1e-5)
    cov_t_sqrt = sqrtm(cov_t + np.eye(cov_t.shape[0]) * 1e-5)
    A = cov_t_sqrt @ np.linalg.inv(cov_s_sqrt)
    
    # Transform source data
    X_s_aligned = X_s_centered @ A.T + X_t_mean
    return X_s_aligned, A

# Align C4 to CARD distribution
X_c4_aligned, coral_transform = coral_alignment(X_c4.values, X_card.values)

# ============================================================================
# STEP 3: Scale with RobustScaler (More Robust to Outliers)
# ============================================================================

# Fit RobustScaler on aligned C4 data
robust_scaler = RobustScaler()
X_c4_scaled = robust_scaler.fit_transform(X_c4_aligned)
X_card_scaled = robust_scaler.transform(X_card.values)

# ============================================================================
# STEP 4: Train Models with Increased Regularization
# ============================================================================

# Split C4 data
X_train, X_val, y_train, y_val = train_test_split(
    X_c4_scaled, y_c4, test_size=0.2, stratify=y_c4, random_state=42
)

# Models with increased regularization for better generalization
models = {
    'Logistic Regression': LogisticRegression(
        C=0.1,  # Increased regularization (lower C = more regularization)
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=10,  # Reduced depth for regularization
        min_samples_split=10,  # Increased for regularization
        min_samples_leaf=5,  # Increased for regularization
        random_state=42,
        class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=5,  # Reduced depth
        learning_rate=0.05,  # Lower learning rate
        reg_alpha=0.1,  # L1 regularization
        reg_lambda=0.1,  # L2 regularization
        random_state=42,
        eval_metric='logloss'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42
    )
}

# Train models
trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    # Evaluate on C4 validation set
    y_pred_c4 = model.predict(X_val)
    y_proba_c4 = model.predict_proba(X_val)[:, 1]
    c4_f1 = f1_score(y_val, y_pred_c4)
    c4_auc = roc_auc_score(y_val, y_proba_c4)
    
    # Evaluate on CARD (external validation)
    y_proba_card = model.predict_proba(X_card_scaled)[:, 1]
    y_pred_card = (y_proba_card >= 0.5).astype(int)
    card_f1 = f1_score(y_card, y_pred_card)
    card_auc = roc_auc_score(y_card, y_proba_card)
    
    print(f"  C4 F1: {c4_f1:.4f}, AUC: {c4_auc:.4f}")
    print(f"  CARD F1: {card_f1:.4f}, AUC: {card_auc:.4f}")
    print(f"  F1 Drop: {c4_f1 - card_f1:.4f}")
    
    trained_models[name] = {
        'model': model,
        'c4_f1': c4_f1,
        'c4_auc': c4_auc,
        'card_f1': card_f1,
        'card_auc': card_auc
    }

# ============================================================================
# STEP 5: Early Stopping Based on CARD Performance
# ============================================================================

# Alternative: Use CARD validation set for early stopping
# This helps prevent overfitting to C4

# ============================================================================
# STEP 6: Save Improved Models
# ============================================================================

output_dir = 'models/card_adapted'
os.makedirs(output_dir, exist_ok=True)

for name, model_info in trained_models.items():
    model_path = os.path.join(output_dir, f'{name.lower().replace(" ", "_")}_card_adapted.joblib')
    joblib.dump(model_info['model'], model_path)
    print(f"Saved {name} to {model_path}")

# Save scaler and CORAL transform
joblib.dump(robust_scaler, os.path.join(output_dir, 'robust_scaler.joblib'))
joblib.dump(coral_transform, os.path.join(output_dir, 'coral_transform.joblib'))

print(f"\\n✅ Models saved to {output_dir}")
print("\\nThese models should have better generalizability to CARD dataset.")
'''

print(retraining_template)

print("\n" + "="*80)
print("KEY DIFFERENCES FROM ORIGINAL C4 TRAINING:")
print("="*80)
print("""
1. CORAL Alignment: Aligns C4 feature distributions to match CARD
2. RobustScaler: More robust to outliers than StandardScaler
3. Increased Regularization: Prevents overfitting to C4
4. Cross-Dataset Evaluation: Monitor performance on both C4 and CARD
5. Early Stopping: Can use CARD validation set to prevent overfitting

EXPECTED IMPROVEMENTS:
- Better generalization to CARD dataset
- Reduced performance drop between C4 and CARD
- More robust to domain shift
- Better handling of outliers and distribution differences
""")

## STEP 11 (DUPLICATE - USE CELL ABOVE): Create Balanced 50/50 Dataset

**⚠️ DUPLICATE CELL - Please use the Step 11 cell above (after Step 10) instead.**

The correct Step 11 cell is located after Step 10 and contains the proper downsampling code.

In [ ]:
# ⚠️ DUPLICATE CELL - This cell has been replaced
# Please use the Step 11 cell above (after Step 10) which contains the correct downsampling code
# That cell filters adolescent/child data and creates a 4053:4053 balanced dataset

print("="*80)
print("⚠️  DUPLICATE CELL - USE STEP 11 ABOVE INSTEAD")
print("="*80)
print("\nThis is a duplicate Step 11 cell with old matching code.")
print("Please use the Step 11 cell located after Step 10 (above) instead.")
print("That cell contains the correct code that:")
print("  1. Filters out adolescent/child questionnaires")
print("  2. Downsamples to create 4053 ASC cases and 4053 non-ASC cases")
print("  3. Saves to card_external_validation_balanced.csv")

## STEP 5: Create Target Variable and Demographics

In [ ]:
print("="*80)
print("CREATING TARGET VARIABLE AND DEMOGRAPHICS")
print("="*80)

# CRITICAL: If autism_target already exists (from Step 11 balancing), preserve it!
# Step 11 creates a balanced 50/50 dataset, and we should NOT overwrite it here
if 'autism_target' in df_card_aggregated.columns:
    print("\n✅ 'autism_target' already exists - preserving balanced target from Step 11")
    print(f"   Current distribution: {df_card_aggregated['autism_target'].value_counts().to_dict()}")
    print(f"   Autism prevalence: {df_card_aggregated['autism_target'].mean()*100:.2f}%")
    print("   ⚠️  Skipping recreation of autism_target to preserve balanced dataset")
else:
    # Create autism_target from diagnosis (only if it doesn't exist)
    print("\n⚠️  'autism_target' not found - creating from diagnosis column...")
    diagnosis_cols = [c for c in df_card_aggregated.columns if 'diagnos' in c.lower() or 'asc' in c.lower()]
    if diagnosis_cols:
        diagnosis_col = diagnosis_cols[0]
        print(f"Using diagnosis column: '{diagnosis_col}'")
        print(f"Sample values: {df_card_aggregated[diagnosis_col].value_counts().head(10).to_dict()}")
        
        # Try different formats
        diagnosis_series = df_card_aggregated[diagnosis_col]
        
        # Check if it's numeric (0/1)
        if diagnosis_series.dtype in ['int64', 'float64']:
            print("  Detected numeric format - treating 1 as autism, 0 as non-autism")
            df_card_aggregated['autism_target'] = (diagnosis_series == 1).astype(int)
        else:
            # Try to convert to numeric first
            diagnosis_numeric = pd.to_numeric(diagnosis_series, errors='coerce')
            if diagnosis_numeric.notna().any() and diagnosis_numeric.nunique() <= 2:
                # If conversion worked and binary, use numeric
                print("  Detected numeric format (after conversion) - treating 1 as autism, 0 as non-autism")
                df_card_aggregated['autism_target'] = (diagnosis_numeric == 1).astype(int)
            else:
                # Text format - look for autism/ASC/ASD keywords
                print("  Detected text format - looking for autism/ASC/ASD keywords")
                diagnosis_str = diagnosis_series.astype(str).str.lower()
                df_card_aggregated['autism_target'] = (
                    diagnosis_str.str.contains('autism|asc|asd|yes|1|true', case=False, na=False) &
                    ~diagnosis_str.str.contains('no|none|0|false|na|nan', case=False, na=False)
                ).astype(int)
        
        print(f"\nAutism target distribution:")
        print(df_card_aggregated['autism_target'].value_counts().to_dict())
        print(f"Autism prevalence: {df_card_aggregated['autism_target'].mean()*100:.2f}%")
    else:
        print("\n⚠️  WARNING: No diagnosis column found - cannot create autism_target")
        print("   Available columns:", [c for c in df_card_aggregated.columns if 'asc' in c.lower() or 'diagnos' in c.lower()])
        df_card_aggregated['autism_target'] = 0

# Age
age_cols = [c for c in df_card_aggregated.columns if 'age' in c.lower()]
if age_cols:
    age_col = age_cols[0]
    df_card_aggregated['age'] = pd.to_numeric(df_card_aggregated[age_col], errors='coerce')
    age_median = df_card_aggregated['age'].median()
    df_card_aggregated['age'] = df_card_aggregated['age'].fillna(age_median)
else:
    df_card_aggregated['age'] = 30
    print("⚠️  Age column not found - using default age=30")

print(f"Age: median={df_card_aggregated['age'].median():.1f}, range={df_card_aggregated['age'].min():.0f}-{df_card_aggregated['age'].max():.0f}")

# Sex
sex_mapping = {
    'male': 1, 'm': 1, '1': 1,
    'female': 2, 'f': 2, '2': 2,
    'other': 3, 'o': 3, '3': 3,
    'prefer not to say': 4, '4': 4
}

sex_cols = [c for c in df_card_aggregated.columns if 'sex' in c.lower() or 'gender' in c.lower()]
if sex_cols:
    sex_col = sex_cols[0]
    df_card_aggregated['sex'] = df_card_aggregated[sex_col].astype(str).str.strip().str.lower().map(sex_mapping).fillna(4)
else:
    df_card_aggregated['sex'] = 4
    print("⚠️  Sex column not found - set to unknown (4)")

df_card_aggregated['sex_num'] = df_card_aggregated['sex'].map({1: 0, 2: 1, 3: 2, 4: 3}).fillna(0).astype(int)
print(f"Sex distribution: {df_card_aggregated['sex'].value_counts().to_dict()}")

# Age groups
df_card_aggregated['age_group_19-30'] = ((df_card_aggregated['age'] >= 19) & (df_card_aggregated['age'] <= 30)).astype(int)
df_card_aggregated['age_group_31-45'] = ((df_card_aggregated['age'] >= 31) & (df_card_aggregated['age'] <= 45)).astype(int)
df_card_aggregated['age_group_46-60'] = ((df_card_aggregated['age'] >= 46) & (df_card_aggregated['age'] <= 60)).astype(int)
df_card_aggregated['age_group_61+'] = (df_card_aggregated['age'] >= 61).astype(int)

# sqrt_age
df_card_aggregated['sqrt_age'] = np.sqrt(np.clip(df_card_aggregated['age'], a_min=0, a_max=None))
print(f"✅ sqrt_age created: range {df_card_aggregated['sqrt_age'].min():.2f}-{df_card_aggregated['sqrt_age'].max():.2f}")

# d_score (SQR - EQ)
df_card_aggregated['d_score'] = df_card_aggregated['sqr_total'] - df_card_aggregated['eq_total']

# age_x_eq interaction
df_card_aggregated['age_x_eq'] = df_card_aggregated['age'] * df_card_aggregated['eq_total']

# eq_sqr_ratio
df_card_aggregated['eq_sqr_ratio'] = df_card_aggregated['eq_total'] / (df_card_aggregated['sqr_total'].replace(0, np.nan) + 1e-8)
df_card_aggregated['eq_sqr_ratio'] = df_card_aggregated['eq_sqr_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Occupation
occupation_cols = [c for c in df_card_aggregated.columns if 'occupation' in c.lower() or 'job' in c.lower()]
if occupation_cols:
    occupation_col = occupation_cols[0]
    df_card_aggregated['is_stem_occupation'] = df_card_aggregated[occupation_col].astype(str).str.contains(
        'science|technology|engineering|math|computer|software|data|research', 
        case=False, na=False
    ).astype(int)
    print(f"✅ is_stem_occupation created: {df_card_aggregated['is_stem_occupation'].sum()} STEM occupations")
else:
    df_card_aggregated['is_stem_occupation'] = 0
    print("⚠️  Occupation column not found - set to 0")

## STEP 6: Feature Alignment to C4 Schema

In [ ]:
print("="*80)
print("FEATURE ALIGNMENT TO C4 SCHEMA")
print("="*80)

# Load C4 feature schema
with open(FEATURE_INFO_PATH, 'r') as f:
    feature_info = json.load(f)

c4_feature_names = feature_info['feature_names']  # 45 features
excluded_features = set(feature_info.get('excluded_features', []))

print(f"\nC4 expects {len(c4_feature_names)} features")
print(f"Excluded features (AQ-related): {len(excluded_features)}")

# Build aligned feature matrix
X_card = pd.DataFrame(index=df_card_aggregated.index)
missing_features = []
available_features = []

for feat_name in c4_feature_names:
    if feat_name in df_card_aggregated.columns:
        X_card[feat_name] = df_card_aggregated[feat_name]
        available_features.append(feat_name)
    else:
        # Missing feature - fill with 0
        X_card[feat_name] = 0
        missing_features.append(feat_name)

# Ensure correct order (CRITICAL)
X_card = X_card[c4_feature_names]

print(f"\n✅ Aligned feature matrix shape: {X_card.shape}")
print(f"✅ Available features: {len(available_features)}/{len(c4_feature_names)}")
print(f"⚠️  Missing features filled with 0: {len(missing_features)}")

if missing_features:
    print(f"\nMissing features (filled with 0):")
    for feat in missing_features[:10]:
        print(f"  - {feat}")
    if len(missing_features) > 10:
        print(f"  ... and {len(missing_features) - 10} more")

# Handle missing values and ensure numeric
X_card = X_card.fillna(0)
for col in X_card.columns:
    if X_card[col].dtype == 'object':
        X_card[col] = pd.to_numeric(X_card[col], errors='coerce').fillna(0)
    else:
        X_card[col] = pd.to_numeric(X_card[col], errors='coerce').fillna(0)

# Check for infinite values
inf_cols = []
for col in X_card.columns:
    if np.isinf(X_card[col]).any():
        inf_cols.append(col)
        X_card[col] = X_card[col].replace([np.inf, -np.inf], 0)

if inf_cols:
    print(f"\n⚠️  Found infinite values in: {inf_cols} (replaced with 0)")
else:
    print(f"\n✅ No infinite values found")

print(f"\n✅ Feature alignment complete - ready for scaling")

## STEP 7: Apply C4 Scaler (NO REFITTING)

In [ ]:
print("="*80)
print("APPLYING C4 SCALER (NO REFITTING)")
print("="*80)

# Load scaler
scaler = joblib.load(SCALER_PATH)

# Apply scaler (fitted on C4, NOT refit on CARD)
X_card_scaled = scaler.transform(X_card.values)

print(f"\n✅ Scaled feature matrix shape: {X_card_scaled.shape}")
print(f"✅ Applied C4 scaler (fitted on C4 training data, NOT refit on CARD)")
print(f"✅ Feature order matches C4 exactly")

# Verify scaling worked
print(f"\nScaled feature statistics:")
print(f"  Mean: {X_card_scaled.mean():.4f}")
print(f"  Std: {X_card_scaled.std():.4f}")
print(f"  Min: {X_card_scaled.min():.4f}")
print(f"  Max: {X_card_scaled.max():.4f}")

## STEP 8: Generate Predictions from All Models

In [ ]:
print("="*80)
print("GENERATING PREDICTIONS FROM ALL MODELS")
print("="*80)

# Load models
loaded_models = {}
for name, path in MODELS.items():
    if os.path.exists(path):
        loaded_models[name] = joblib.load(path)

# IMPORTANT: Using default 0.5 threshold to match C4 evaluation methodology
# C4 models were evaluated using model.predict() which uses 0.5 threshold by default
# This ensures fair comparison with C4 training results
print("\n⚠️  Using default 0.5 threshold (matching C4 evaluation methodology)")
print("   C4 results in original_dataset_results.json were generated with:")
print("   model.predict() → default 0.5 threshold (sklearn standard)")

# Generate predictions
predictions = {}
probabilities = {}

for name, model in loaded_models.items():
    y_proba = model.predict_proba(X_card_scaled)[:, 1]
    # Use 0.5 threshold to match C4 evaluation (model.predict() uses 0.5)
    y_pred = (y_proba >= 0.5).astype(int)
    predictions[name] = y_pred
    probabilities[name] = y_proba
    print(f"\n{name}:")
    print(f"  Generated predictions for {len(y_pred)} samples")
    print(f"  Probability range: {y_proba.min():.4f} - {y_proba.max():.4f}")
    print(f"  Predicted positives: {y_pred.sum()} ({y_pred.mean()*100:.1f}%)")
    print(f"  Threshold: 0.5 (default, matching C4 evaluation)")

print("\n✅ Predictions generated from all models using default 0.5 threshold")
print("   This matches the C4 evaluation methodology exactly")

## STEP 9: Evaluate Performance

In [ ]:
print("="*80)
print("EVALUATING PERFORMANCE")
print("="*80)

if 'autism_target' in df_card_aggregated.columns:
    y_true = df_card_aggregated['autism_target'].values
    
    print(f"\nGround truth available:")
    print(f"  Total samples: {len(y_true)}")
    print(f"  Autism cases: {y_true.sum()} ({y_true.mean()*100:.2f}%)")
    print(f"  Non-autism cases: {(y_true == 0).sum()} ({(y_true == 0).mean()*100:.2f}%)")
    
    results = {}
    for name in loaded_models.keys():
        y_pred = predictions[name]
        y_proba = probabilities[name]
        
        results[name] = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_proba)
        }
    
    metrics_df = pd.DataFrame(results).T
    
    print("\n" + "="*80)
    print("EXTERNAL VALIDATION RESULTS ON CARD")
    print("="*80)
    print(metrics_df.round(4).sort_values('auc', ascending=False))
    
    # Compare with C4 performance
    print("\n" + "="*80)
    print("COMPARISON WITH C4 PERFORMANCE")
    print("="*80)
    
    c4_results_path = os.path.join(ARTIFACT_DIR, 'original_dataset_results.json')
    if os.path.exists(c4_results_path):
        with open(c4_results_path, 'r') as f:
            c4_results = json.load(f)
        
        comparison_data = []
        for model_name in metrics_df.index:
            if model_name in c4_results:
                comparison_data.append({
                    'Model': model_name,
                    'C4_F1': c4_results[model_name]['f1'],
                    'CARD_F1': metrics_df.loc[model_name, 'f1'],
                    'C4_AUC': c4_results[model_name]['auc'],
                    'CARD_AUC': metrics_df.loc[model_name, 'auc'],
                    'F1_Drop': c4_results[model_name]['f1'] - metrics_df.loc[model_name, 'f1'],
                    'AUC_Drop': c4_results[model_name]['auc'] - metrics_df.loc[model_name, 'auc']
                })
        
        comparison_df = pd.DataFrame(comparison_data)
        print(comparison_df.round(4))
    
    # Save results
    metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'card_external_validation_results.csv'))
    print(f"\n✅ Results saved to: {OUTPUT_DIR}/card_external_validation_results.csv")
    
else:
    print("\n⚠️  No ground truth labels available - skipping evaluation")
    print("   Predictions will be saved but metrics cannot be calculated")

## STEP 10: Save Predictions and Metadata

In [ ]:
print("="*80)
print("SAVING PREDICTIONS AND METADATA")
print("="*80)

# Save predictions
pred_df = pd.DataFrame({
    'userid': df_card_aggregated['userid'].values if 'userid' in df_card_aggregated.columns else np.arange(len(df_card_aggregated))
})

for name in loaded_models.keys():
    pred_df[f'proba_{name.replace(" ", "_").lower()}'] = probabilities[name]
    pred_df[f'pred_{name.replace(" ", "_").lower()}'] = predictions[name]

# Add ground truth if available
if 'autism_target' in df_card_aggregated.columns:
    pred_df['autism_target'] = df_card_aggregated['autism_target'].values

pred_path = os.path.join(OUTPUT_DIR, 'card_external_predictions.csv')
pred_df.to_csv(pred_path, index=False)
print(f"\n✅ Predictions saved to: {pred_path}")

# Save feature alignment info
alignment_info = {
    'c4_feature_count': len(c4_feature_names),
    'card_available_features': len(available_features),
    'card_missing_features': len(missing_features),
    'missing_features': missing_features,
    'available_features': available_features,
    'excluded_features': list(excluded_features),
    'note': 'CARD dataset has SPQ data (unlike YBT). AQ features excluded to prevent data leakage.'
}

alignment_path = os.path.join(OUTPUT_DIR, 'card_feature_alignment.json')
with open(alignment_path, 'w') as f:
    json.dump(alignment_info, f, indent=2)
print(f"✅ Feature alignment info saved to: {alignment_path}")

print("\n" + "="*80)
print("EXTERNAL VALIDATION COMPLETE")
print("="*80)
print(f"\nSummary:")
print(f"  Dataset: CARD")
print(f"  Participants: {len(df_card_aggregated)}")
print(f"  Features: {X_card_scaled.shape[1]} (aligned to C4)")
print(f"  Models tested: {len(loaded_models)}")
if 'autism_target' in df_card_aggregated.columns:
    best_model = metrics_df['auc'].idxmax()
    print(f"  Best model: {best_model} (AUC: {metrics_df.loc[best_model, 'auc']:.4f})")
print(f"\n✅ All outputs saved to: {OUTPUT_DIR}")